# Phase 3: VLM counting (systems B, C, D)

Qwen2.5-VL-7B-Instruct, 4-bit, on the fixed queries from the 300-image subset (whole image; each image's most frequent class, plus car).
- **B direct:** one integer. **C localize-then-count:** one point per object. **D grounded:** the detector's boxes drawn and listed, then one integer.
- Strict parsing: an unparsable answer is kept and scored as wrong.
- The order is shuffled once with a fixed seed and each query runs B, D, C in turn, so stopping early leaves a random sample with all systems paired.
- Results are written after every answer. The time box stops new queries at `TIME_H`.

## How to run (about 3 h at most: the time box stops new queries after 2.5 h; timing is unknown until this run)
- **Inputs:** the same weights Dataset as `phase1_eval` (`yolo26m_tile_visdrone.pt`, `yolo26m_tile_uavdt.pt`). On a second account, share the Dataset with it first.
- **Secret:** `CARPK_PASSWORD`, attached to this notebook.
- **Settings:** GPU T4 x2, Internet ON. *Save Version → Save & Run All*. It can run at the same time as `phase1_eval`, on the other account.
- The first cells print a few real answers. If they look broken, stop and send them.
- **Send back** `/kaggle/working/phase3_vlm/send_back.zip`.

**Rerun (10 Nov 2026).** The first full run lost 115 of 346 queries to GPU out-of-memory errors: the notebook's own detector run left memory
held on GPU 0, and an out-of-memory error was not freed before the next query. `ONLY_QIDS` now reruns just those queries, with the same model,
prompts, image size and detector boxes; the GPU is freed before the workers start, and a query that still runs out of memory is retried on a
smaller image (the size used is logged per answer). About 45 minutes.

In [ ]:
# --- Install (Kaggle) ------------------------------------------------------
from pathlib import Path
if Path("/kaggle").exists():
    !pip install -q "ultralytics==8.4.174" "sahi==0.12.8" pycocotools ensemble-boxes pytest gdown "transformers>=4.45" accelerate bitsandbytes qwen-vl-utils
Path("dronekit_src/dronekit").mkdir(parents=True, exist_ok=True)
Path("dronekit_src/tests").mkdir(parents=True, exist_ok=True)

## dronekit package (generated from the repo by `tools/`; do not edit here)

In [ ]:
%%writefile dronekit_src/dronekit/__init__.py
"""dronekit: shared data loading, inference engine, scorer and plots for the drone-QA project.

One implementation of the evaluation protocol, imported by every notebook.
"""
__version__ = "0.1.0"

VISDRONE_CLASSES = ["pedestrian", "people", "bicycle", "car", "van",
                    "truck", "tricycle", "awning-tricycle", "bus", "motor"]

In [ ]:
%%writefile dronekit_src/dronekit/conformal.py
"""Conformal intervals for per-image object counts, calibrated by video sequence.

Point count yhat from detections at a dev-tuned threshold; score s(y, yhat) measures the count error.

Exchangeability: frames of one video are not exchangeable, but whole sequences of a split can be treated as exchangeable (each is a
separate flight). So the calibration unit is the sequence. Three methods:
- `sequence` (primary): each sequence contributes one score, its `seq_q` within-sequence quantile (default 1 - alpha). If a test sequence
  is exchangeable with the calibration sequences, then with probability >= 1 - alpha at least a `seq_q` share of its frames are covered.
  Rigorous and conservative: it needs ceil((K+1)(1-alpha)) <= K, so at least 9 sequences for alpha 0.1 and 4 for alpha 0.2.
- `pooled` (secondary): every sequence gets the same total weight, plus one unit of weight at +inf, and the threshold is the weighted
  1 - alpha quantile (hierarchical conformal prediction, Lee, Barber and Willett). Targets the coverage of a random frame of a new sequence;
  the worst-case bound is 1 - 2 alpha, not 1 - alpha.
- `image` (baseline): every frame exchangeable. Expected to under-cover.
"""
from __future__ import annotations

import numpy as np

SCORES = ("abs", "sqrt")      # |y - yhat| and |y - yhat| / sqrt(yhat + 1)
METHODS = ("sequence", "pooled", "image")


def score(y, yhat, kind="sqrt"):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    e = np.abs(y - yhat)
    return e if kind == "abs" else e / np.sqrt(yhat + 1)


def interval(yhat, q, kind="sqrt"):
    """Integer count interval [lo, hi] holding every y with score(y, yhat) <= q. Lower end is at least 0."""
    yhat = np.asarray(yhat, float)
    r = q if kind == "abs" else q * np.sqrt(yhat + 1)
    return np.maximum(np.ceil(yhat - r - 1e-9), 0), np.floor(yhat + r + 1e-9)


def conformal_quantile(scores, alpha):
    """The ceil((n+1)(1-alpha))-th smallest score; +inf when n is too small for that rank to exist."""
    s = np.sort(np.asarray(scores, float))
    n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(s[k - 1]) if 1 <= k <= n else float("inf")


def sequence_scores(s, groups, seq_q):
    """One score per sequence: the `seq_q` quantile of its frame scores (upper order statistic, so never interpolated down)."""
    s, groups = np.asarray(s, float), np.asarray(groups)
    return np.array([np.quantile(s[groups == g], seq_q, method="higher") for g in np.unique(groups)], float)


def pooled_quantile(s, groups, alpha):
    """Weighted 1 - alpha quantile: each of K sequences has total weight 1/(K+1), split over its frames; +inf holds the last 1/(K+1)."""
    s, g = np.asarray(s, float), np.asarray(groups)
    uniq, inv, cnt = np.unique(g, return_inverse=True, return_counts=True)
    w = 1.0 / (cnt[inv] * (len(uniq) + 1))
    o = np.argsort(s)
    c = np.cumsum(w[o])
    i = np.searchsorted(c, (1 - alpha) - 1e-12)
    return float(s[o[i]]) if i < len(s) else float("inf")


def calibrate(s, groups=None, alpha=0.1, method="sequence", seq_q=None):
    """Threshold q for one of METHODS."""
    if method == "sequence":
        return conformal_quantile(sequence_scores(s, groups, 1 - alpha if seq_q is None else seq_q), alpha)
    if method == "pooled":
        return pooled_quantile(s, groups, alpha)
    return conformal_quantile(s, alpha)


def mondrian(s, strata, groups=None, alpha=0.1, method="sequence", seq_q=None):
    """{stratum: q}, calibrated separately in each stratum. A stratum whose calibration data cannot support alpha gets +inf, which
    is reported as an uninformative interval rather than hidden by borrowing from other strata."""
    s, strata = np.asarray(s, float), np.asarray(strata)
    g = None if groups is None else np.asarray(groups)
    out = {}
    for k in np.unique(strata):
        m = strata == k
        out[k.item() if hasattr(k, "item") else k] = calibrate(s[m], None if g is None else g[m], alpha, method, seq_q)
    return out


def poisson_interval(yhat, alpha=0.1):
    """Baseline: central Poisson interval with mean yhat (no coverage guarantee)."""
    from scipy.stats import poisson
    yhat = np.asarray(yhat, float)
    mu = np.maximum(yhat, 1e-9)
    return poisson.ppf(alpha / 2, mu), poisson.ppf(1 - alpha / 2, mu)


def residual_range(yhat, cal_y, cal_yhat, lo=0.1, hi=0.9, eps=1.0):
    """Baseline: the 10th-90th percentile of relative residuals on calibration records, applied to yhat (no coverage guarantee)."""
    rel = (np.asarray(cal_y, float) - np.asarray(cal_yhat, float)) / (np.asarray(cal_yhat, float) + eps)
    a, b = np.quantile(rel, [lo, hi])
    yhat = np.asarray(yhat, float)
    return np.maximum(np.ceil(yhat + a * (yhat + eps) - 1e-9), 0), np.floor(yhat + b * (yhat + eps) + 1e-9)


def coverage(y, lo, hi):
    y = np.asarray(y, float)
    return float(np.mean((y >= lo) & (y <= hi)))


def mean_width(lo, hi):
    """Mean of hi - lo. Infinite intervals make it infinite, on purpose."""
    return float(np.mean(np.asarray(hi, float) - np.asarray(lo, float)))


def sequence_coverage(y, lo, hi, groups):
    """Coverage per sequence: {sequence: coverage}. Pooled coverage can hide sequences that fail."""
    cov = (np.asarray(y, float) >= lo) & (np.asarray(y, float) <= hi)
    g = np.asarray(groups)
    return {k: float(cov[g == k].mean()) for k in np.unique(g)}


def bootstrap_ci(values, groups, stat=np.mean, n=2000, seed=0, level=0.95):
    """Percentile interval from resampling whole sequences."""
    v, g = np.asarray(values, float), np.asarray(groups)
    uniq = np.unique(g)
    parts = [v[g == u] for u in uniq]
    rng = np.random.default_rng(seed)
    b = [stat(np.concatenate([parts[i] for i in rng.integers(0, len(uniq), len(uniq))])) for _ in range(n)]
    return tuple(np.quantile(b, [(1 - level) / 2, (1 + level) / 2]))


def few_shot_draws(frames_by_seq, k, n_draws=20, seed=0):
    """Draws of about k labeled target frames taken from whole sequences, in random sequence order: sequences are added until at
    least k frames are held, and only the first k frames of the last sequence are used. Returns one list of (sequence, n frames)
    per draw. Whole sequences are then excluded from evaluation in that draw."""
    rng = np.random.default_rng(seed)
    seqs = sorted(frames_by_seq)
    out = []
    for _ in range(n_draws):
        pick, left = [], k
        for i in rng.permutation(len(seqs)):
            if left <= 0:
                break
            s = seqs[i]
            t = min(left, frames_by_seq[s])
            pick.append((s, t))
            left -= t
        out.append(pick)
    return out

In [ ]:
%%writefile dronekit_src/dronekit/countset.py
"""Count set: one record per (image, class or group, region), with the exact ground-truth count and stratification variables.

Rules (Phase 2 plan): an object counts toward a region if its box center lies inside; objects centered in an ignore region are excluded;
records where ignore regions cover more than 20% of the region are dropped. Size and density tertiles are computed on the dev split and
frozen. Records hold an image's boxes through counts only, so the same functions count detections (`count_dets`).
"""
from __future__ import annotations

import numpy as np

REGIONS = {"whole": (0, 0, 1, 1), "left": (0, 0, .5, 1), "right": (.5, 0, 1, 1),
           "q1": (0, 0, .5, .5), "q2": (.5, 0, 1, .5), "q3": (0, .5, .5, 1), "q4": (.5, .5, 1, 1)}
GROUPS = {"people": ["pedestrian", "people"],
          "vehicles": ["car", "van", "truck", "bus", "tricycle", "awning-tricycle", "motor"]}
MAX_IGNORE = 0.20
COUNT_BINS = [(0, 0, "0"), (1, 8, "1-8"), (9, 20, "9-20"), (21, 50, "21-50"), (51, 10 ** 9, ">50")]
SEED = 0


def count_bin(n: int) -> str:
    return next(lbl for a, b, lbl in COUNT_BINS if a <= n <= b)


def targets(class_names):
    """{target name: class ids}. Every class, plus a group when at least two of its classes exist in this dataset."""
    out = {c: [i] for i, c in enumerate(class_names)}
    for g, members in GROUPS.items():
        ids = [class_names.index(m) for m in members if m in class_names]
        if len(ids) >= 2:
            out[g] = ids
    return out


def region_box(name, W, H):
    a, b, c, d = REGIONS[name]
    return a * W, b * H, c * W, d * H


def centers(b):
    return (b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2


def in_region(b, box):
    """Half-open, so that a center on a shared border belongs to exactly one region."""
    cx, cy = centers(b)
    return (cx >= box[0]) & (cx < box[2]) & (cy >= box[1]) & (cy < box[3])


def in_ignore(b, ignore):
    if not len(b) or not len(ignore):
        return np.zeros(len(b), bool)
    cx, cy = centers(b)
    return ((cx[:, None] >= ignore[None, :, 0]) & (cx[:, None] < ignore[None, :, 2]) &
            (cy[:, None] >= ignore[None, :, 1]) & (cy[:, None] < ignore[None, :, 3])).any(1)


def ignore_masks(rec, step=4):
    """{region: share of the region inside ignore regions (union)}, on a coarse grid."""
    W, H = rec.width, rec.height
    m = np.zeros((int(np.ceil(H / step)), int(np.ceil(W / step))), bool)
    for x1, y1, x2, y2 in rec.ignore:
        m[max(0, int(y1 // step)):max(0, int(np.ceil(y2 / step))), max(0, int(x1 // step)):max(0, int(np.ceil(x2 / step)))] = True
    out = {}
    for k in REGIONS:
        x1, y1, x2, y2 = region_box(k, m.shape[1], m.shape[0])
        out[k] = float(m[int(y1):int(y2), int(x1):int(x2)].mean())
    return out


def density(b):
    """Mean nearest-neighbor center distance between counted objects, in median box sides. NaN below 2 objects."""
    if len(b) < 2:
        return float("nan")
    cx, cy = centers(b)
    d2 = (cx[:, None] - cx[None, :]) ** 2 + (cy[:, None] - cy[None, :]) ** 2
    np.fill_diagonal(d2, np.inf)
    side = np.median(np.sqrt((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])))
    return float(np.sqrt(d2.min(1)).mean() / max(side, 1e-6))


def records(split):
    """Ground-truth records for a Split, as a list of dicts."""
    tg = targets(split.class_names)
    rows = []
    for r in split.images:
        keep = ~in_ignore(r.boxes[:, 1:5], r.ignore)
        B = r.boxes[keep]
        ign = ignore_masks(r)
        for reg in REGIONS:
            if ign[reg] > MAX_IGNORE:
                continue
            box = region_box(reg, r.width, r.height)
            inside = B[in_region(B[:, 1:5], box)]
            for t, ids in tg.items():
                b = inside[np.isin(inside[:, 0], ids)]
                n = len(b)
                sides = np.sqrt((b[:, 3] - b[:, 1]) * (b[:, 4] - b[:, 2])) if n else np.zeros(0)
                rows.append(dict(dataset=split.dataset, split=split.name, image=r.key, sequence=r.sequence, target=t, region=reg,
                                 count=n, count_bin=count_bin(n), median_side=float(np.median(sides)) if n else float("nan"),
                                 median_area_frac=float(np.median(sides ** 2) / (r.width * r.height)) if n else float("nan"),
                                 density=density(b[:, 1:5]),
                                 frac_occluded=float((b[:, 5] > 0).mean()) if n and (b[:, 5] >= 0).all() else float("nan"),
                                 frac_truncated=float((b[:, 6] > 0).mean()) if n and (b[:, 6] >= 0).all() else float("nan"),
                                 altitude=r.attrs.get("altitude"), weather=r.attrs.get("weather")))
    return rows


def tertile_edges(values):
    """Two cut points from dev data; frozen and reused on every other split."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    return [float(x) for x in np.quantile(v, [1 / 3, 2 / 3])]


def tertile(values, edges, labels=("low", "mid", "high")):
    v = np.asarray(values, float)
    out = np.full(len(v), "none", dtype=object)
    ok = np.isfinite(v)
    out[ok] = np.array(labels, dtype=object)[np.searchsorted(edges, v[ok], side="right")]
    return out


def count_dets(d, image_size, ignore, class_ids, region="whole", thresholds=None):
    """Predicted count for one image from detections (N,6), by the same rules as the ground truth.
    thresholds: {class id: minimum score}; classes not listed are not counted."""
    if not len(d):
        return 0
    box = region_box(region, *image_size)
    m = np.isin(d[:, 5].astype(int), class_ids)
    if thresholds is not None:
        m &= d[:, 4] >= np.array([thresholds.get(int(c), np.inf) for c in d[:, 5]])
    b = d[m][:, :4]
    return int((in_region(b, box) & ~in_ignore(b, ignore)).sum())


def stratified_subset(df, n=300, seed=SEED, by=("dataset", "count_bin", "size_bin")):
    """Fixed image subset balanced across dataset x count bin x size bin, using each image's whole-image record of its most frequent
    target. Cells are filled in turn until n images are held. Returns the sorted (dataset, image) list."""
    rng = np.random.default_rng(seed)
    w = df[(df["region"] == "whole") & (df["count"] > 0)]
    w = w.sort_values("count", ascending=False, kind="stable").drop_duplicates(["dataset", "image"])
    cells = {k: list(g["image"]) for k, g in w.groupby(list(by), sort=True)}
    for k in cells:
        rng.shuffle(cells[k])
    seen, out, active = set(), [], sorted(cells)
    while len(out) < n and active:
        for k in list(active):
            while cells[k] and (k[0], cells[k][-1]) in seen:
                cells[k].pop()
            if not cells[k]:
                active.remove(k)
                continue
            im = cells[k].pop()
            seen.add((k[0], im)); out.append((k[0], im))
            if len(out) == n:
                break
    return sorted(out)


THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)


def _kept_scores(rec, d, cid):
    d = d[d[:, 5].astype(int) == cid]
    return d[~in_ignore(d[:, :4], rec.ignore), 4] if len(d) else np.zeros(0)


def tune_thresholds(split, dets, class_ids):
    """{class id: threshold with the lowest whole-image count MAE on this (dev) split}. GT and detections follow the record rules:
    centered in an ignore region = not counted. Ties go to the lower threshold, and images with heavily ignored regions stay out."""
    ign = [ignore_masks(r)["whole"] <= MAX_IGNORE for r in split.images]
    out = {}
    for c in class_ids:
        y, sc = [], []
        for r, ok in zip(split.images, ign):
            if ok:
                b = r.boxes[r.boxes[:, 0] == c]
                y.append(int((~in_ignore(b[:, 1:5], r.ignore)).sum()))
                sc.append(np.sort(_kept_scores(r, dets.get(r.key, np.zeros((0, 6))), c)))
        y = np.array(y)
        n = np.array([[len(s) - np.searchsorted(s, t, side="left") for t in THRESHOLD_GRID] for s in sc]) if sc else np.zeros((0, len(THRESHOLD_GRID)))
        out[int(c)] = float(THRESHOLD_GRID[int(np.argmin(np.abs(n - y[:, None]).mean(0)))]) if len(y) else 0.5
    return out


def _side_density(b):
    if not len(b):
        return float("nan"), float("nan")
    return float(np.median(np.sqrt((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])))), density(b)


def paired_records(split, dets, thresholds, pred_spec, gt_spec, regions=("whole",)):
    """One row per (image, target, region): true count y, predicted count yhat, ground-truth strata variables and the same variables
    from the detections (usable at test time). pred_spec / gt_spec: {target: class ids of the model / of the split's labels}."""
    rows = []
    for r in split.images:
        ign = ignore_masks(r)
        B = r.boxes[~in_ignore(r.boxes[:, 1:5], r.ignore)]
        d = dets.get(r.key, np.zeros((0, 6), np.float32))
        keep = np.array([d_[4] >= thresholds.get(int(d_[5]), np.inf) for d_ in d], bool) if len(d) else np.zeros(0, bool)
        D = d[keep]
        D = D[~in_ignore(D[:, :4], r.ignore)] if len(D) else D
        for reg in regions:
            if ign[reg] > MAX_IGNORE:
                continue
            box = region_box(reg, r.width, r.height)
            Bi, Di = B[in_region(B[:, 1:5], box)], D[in_region(D[:, :4], box)]
            for t in pred_spec:
                g = Bi[np.isin(Bi[:, 0], gt_spec[t])]
                p = Di[np.isin(Di[:, 5].astype(int), pred_spec[t])]
                gs, gd = _side_density(g[:, 1:5])
                ps, pd_ = _side_density(p[:, :4])
                rows.append(dict(dataset=split.dataset, split=split.name, image=r.key, sequence=r.sequence, target=t, region=reg, y=len(g), yhat=len(p), count_bin=count_bin(len(g)),
                                 gt_side=gs, gt_density=gd, pred_side=ps, pred_density=pd_, altitude=r.attrs.get("altitude")))
    return rows

In [ ]:
%%writefile dronekit_src/dronekit/data.py
"""Dataset loading into one common structure.

Every dataset becomes a `Split`: a list of `ImageRecord`s, each holding
  boxes   (N, 7) float32: class_id (0-based), x1, y1, x2, y2, occlusion, truncation
  ignore  (M, 4) float32: x1, y1, x2, y2 of ignore regions
  sequence: video-sequence id parsed from the filename (bootstrap / CV grouping unit)

VisDrone rules: category 0 or score 0 = ignore region; category 11
("others") dropped; categories 1-10 -> class ids 0-9. Boxes are kept exactly as annotated
(not clipped), because the eval-v2 scorer used the raw x, y, w, h.
"""
from __future__ import annotations

import io
import urllib.request
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from PIL import Image

from . import VISDRONE_CLASSES

VISDRONE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-{split}.zip"


@dataclass
class ImageRecord:
    key: str                 # filename stem, unique within a split
    path: Path
    width: int
    height: int
    sequence: str
    boxes: np.ndarray = field(default_factory=lambda: np.zeros((0, 7), np.float32))
    ignore: np.ndarray = field(default_factory=lambda: np.zeros((0, 4), np.float32))
    attrs: dict = field(default_factory=dict)          # sequence-level attributes (UAVDT: weather, altitude, view)


@dataclass
class Split:
    dataset: str
    name: str
    class_names: list
    images: list             # list[ImageRecord], sorted by key; position = COCO image id

    def __len__(self):
        return len(self.images)

    def subset(self, n):
        """First n images (smoke tests). Image ids stay positional."""
        return Split(self.dataset, self.name, self.class_names, self.images[:n])

    @property
    def n_objects(self):
        return int(sum(len(r.boxes) for r in self.images))

    @property
    def sequences(self):
        return sorted({r.sequence for r in self.images})


def visdrone_sequence(stem: str) -> str:
    """'0000001_02999_d_0000005' -> '0000001'. VisDrone-DET frames from one video share the first field."""
    return stem.split("_")[0]


def parse_visdrone_txt(text: str, training_rules: bool = False):
    """Return (boxes (N,7), ignore (M,4)) from one VisDrone annotation file.

    The two original notebooks differ on one edge case, and both are kept on purpose:
    - scorer (evaluation v2, default): category 11 rows are dropped first, even with score 0;
    - training converter (visdrone-training, training_rules=True): score 0 makes any row an ignore
      region (painted gray), including category 11, which is checked afterwards.
    """
    boxes, ignore = [], []
    for row in text.strip().splitlines():
        v = row.strip().rstrip(",").split(",")
        if len(v) < 6:
            continue
        x, y, w, h, score, cat = map(int, v[:6])
        trunc = int(v[6]) if len(v) > 6 else -1
        occ = int(v[7]) if len(v) > 7 else -1
        if w <= 0 or h <= 0 or (cat == 11 and not training_rules):
            continue
        if score == 0 or cat == 0:
            ignore.append([x, y, x + w, y + h])
        elif 1 <= cat <= 10:
            boxes.append([cat - 1, x, y, x + w, y + h, occ, trunc])
    return (np.array(boxes, np.float32).reshape(-1, 7), np.array(ignore, np.float32).reshape(-1, 4))


def load_visdrone(split_dir, split_name: str | None = None, training_rules: bool = False) -> Split:
    """Load a VisDrone2019-DET split folder (with images/ and annotations/).
    training_rules=True reproduces the training converter (see parse_visdrone_txt); use it only for building
    training data, never for scoring."""
    split_dir = Path(split_dir)
    recs = []
    for ann in sorted((split_dir / "annotations").glob("*.txt")):
        img = split_dir / "images" / f"{ann.stem}.jpg"
        with Image.open(img) as im:
            W, H = im.size
        boxes, ignore = parse_visdrone_txt(ann.read_text(), training_rules)
        recs.append(ImageRecord(ann.stem, img, W, H, visdrone_sequence(ann.stem), boxes, ignore))
    return Split("visdrone", split_name or split_dir.name, list(VISDRONE_CLASSES), recs)


def download_visdrone(root, split: str) -> Path:
    """Download + extract one split ('train', 'val', 'test-dev') into root. Returns the split folder."""
    root = Path(root)
    d = root / f"VisDrone2019-DET-{split}"
    if not (d / "annotations").exists():
        root.mkdir(parents=True, exist_ok=True)
        print("Downloading", d.name)
        with urllib.request.urlopen(VISDRONE_URL.format(split=split)) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(root)
    return d


def split_stats(split: Split) -> dict:
    """Object counts and size shares (non-ignored objects)."""
    b = np.concatenate([r.boxes for r in split.images]) if split.images else np.zeros((0, 7))
    area = (b[:, 3] - b[:, 1]) * (b[:, 4] - b[:, 2])
    n = max(len(b), 1)
    out = dict(images=len(split), sequences=len(split.sequences), objects=len(b),
               objects_per_image=len(b) / max(len(split), 1),
               tiny=float((area < 16 ** 2).sum() / n),
               small=float(((area >= 16 ** 2) & (area < 32 ** 2)).sum() / n),
               medium_plus=float((area >= 32 ** 2).sum() / n))
    for k, c in enumerate(split.class_names):
        out[f"share_{c}"] = float((b[:, 0] == k).sum() / n)
    return out


# --------------------------------------------------------------------------- UAVDT
UAVDT_CLASSES = ["car", "truck", "bus"]
UAVDT_ATTRS = ["daylight", "night", "fog", "low-alt", "medium-alt", "high-alt",
               "front-view", "side-view", "bird-view", "long-term"]
# gt_whole.txt codes -> ordinal levels (0 = none). Occlusion: 1 no, 4 small, 3 medium, 2 large.
_UAVDT_OCC = {1: 0, 4: 1, 3: 2, 2: 3}
_UAVDT_OUT = {1: 0, 3: 1, 2: 2}                         # out-of-view: 1 no, 3 small, 2 medium


def read_uavdt_attrs(attr_root):
    r"""M_attr/{train,test}/M0101_attr.txt -> ({seq: attrs dict}, {seq: 'train'|'test'}).
    One official file is named 'M0701 _attr.txt' (stray space), hence the \s*."""
    import re
    attrs, split = {}, {}
    for p in Path(attr_root).rglob("*_attr.txt"):
        m = re.match(r"(M\d+)\s*_attr\.txt$", p.name)
        if not m:
            continue
        bits = [int(b) for b in p.read_text().strip().split(",")]
        a = dict(zip(UAVDT_ATTRS, bits))
        a["weather"] = next((n for n in ("daylight", "night", "fog") if a.get(n)), "unknown")
        a["altitude"] = next((n for n in ("low-alt", "medium-alt", "high-alt") if a.get(n)), "unknown")
        attrs[m[1]], split[m[1]] = a, p.parent.name
    return attrs, split


def _uavdt_rows(path):
    rows = [list(map(int, l.split(","))) for l in Path(path).read_text().splitlines() if l.strip()]
    return np.array(rows, np.int64).reshape(-1, 9)


def find_uavdt_frames(images_root, seq):
    """{frame_index: path} for one sequence; files are img000001.jpg ... (frame 1 = img000001)."""
    import re
    d = next((p for p in Path(images_root).rglob(seq) if p.is_dir()), None)
    if d is None:
        return {}
    return {int(m[1]): p for p in d.glob("img*.jpg") if (m := re.match(r"img(\d+)\.jpg$", p.name))}


def load_uavdt(images_root, gt_dir, attr_root, seqs, stride=1, name="uavdt", size=None):
    """UAVDT-M detection frames of `seqs`, every `stride`-th annotated frame (frames 1, 1+stride, ...).

    boxes columns 5/6 = occlusion / out-of-view as ordinal levels (0 none ... 3 large).
    Boxes with category outside 1-3 are dropped; *_gt_ignore.txt boxes become ignore regions.
    `size` = (W, H) skips opening images; otherwise read from the first frame of each sequence (widths differ: M0901 is 960).
    """
    attrs, _ = read_uavdt_attrs(attr_root)
    recs = []
    for seq in seqs:
        gt = _uavdt_rows(Path(gt_dir) / f"{seq}_gt_whole.txt")
        ign_p = Path(gt_dir) / f"{seq}_gt_ignore.txt"
        ign = _uavdt_rows(ign_p) if ign_p.exists() else np.zeros((0, 9), np.int64)
        frames_img = find_uavdt_frames(images_root, seq)
        # annotated frames only: image files without a gt_whole row are unlabeled, not known to be empty
        frames = sorted(set(gt[:, 0].tolist()))
        frames = [f for f in frames if (f - frames[0]) % stride == 0] if frames else []
        W, H = size or (None, None)
        if W is None and frames_img:
            with Image.open(next(iter(frames_img.values()))) as im:
                W, H = im.size
        for f in frames:
            g = gt[(gt[:, 0] == f) & np.isin(gt[:, 8], [1, 2, 3])]
            b = np.c_[g[:, 8] - 1, g[:, 2], g[:, 3], g[:, 2] + g[:, 4], g[:, 3] + g[:, 5],
                      [_UAVDT_OCC.get(int(v), -1) for v in g[:, 7]], [_UAVDT_OUT.get(int(v), -1) for v in g[:, 6]]]
            g = ign[ign[:, 0] == f]
            ig = np.c_[g[:, 2], g[:, 3], g[:, 2] + g[:, 4], g[:, 3] + g[:, 5]]
            path = frames_img.get(f, Path(images_root) / seq / f"img{f:06d}.jpg")
            recs.append(ImageRecord(f"{seq}_img{f:06d}", path, W, H, seq,
                                    np.asarray(b, np.float32).reshape(-1, 7), np.asarray(ig, np.float32).reshape(-1, 4),
                                    dict(attrs.get(seq, {}))))
    return Split("uavdt", name, list(UAVDT_CLASSES), recs)


# --------------------------------------------------------------------------- CARPK
CARPK_CLASSES = ["car"]


def load_carpk(root, split_name="test"):
    """CARPK devkit: Images/*.png, Annotations/*.txt (one 'x1 y1 x2 y2 cls' per line), ImageSets/{train,test}.txt.
    Sequence id = parking lot (filename without the trailing frame number)."""
    root = Path(root)
    img_dir = next(p for p in root.rglob("Images") if p.is_dir())
    ann_dir = next(p for p in root.rglob("Annotations") if p.is_dir())
    lst = next((p for p in root.rglob(f"{split_name}.txt") if p.parent.name == "ImageSets"), None)
    stems = [s.strip() for s in lst.read_text().split()] if lst else sorted(p.stem for p in img_dir.glob("*.png"))
    recs = []
    for s in sorted(stems):
        img = img_dir / f"{s}.png"
        with Image.open(img) as im:
            W, H = im.size
        rows = []
        a = ann_dir / f"{s}.txt"
        if a.exists():
            for line in a.read_text().splitlines():
                v = line.replace(",", " ").split()
                if len(v) >= 4:
                    x1, y1, x2, y2 = map(float, v[:4])
                    if x2 > x1 and y2 > y1:
                        rows.append([0, x1, y1, x2, y2, -1, -1])
        recs.append(ImageRecord(s, img, W, H, s.rsplit("_", 1)[0], np.array(rows, np.float32).reshape(-1, 7)))
    return Split("carpk", split_name, list(CARPK_CLASSES), recs)


# --------------------------------------------------------------------------- views of a split
def without_ignore(split):
    """Same images and boxes with no ignore regions: detections on them then count as false positives,
    as in the Ultralytics validator. Diagnostics only; never report it as an eval-v2 number."""
    import copy
    out = []
    for r in split.images:
        rr = copy.copy(r)
        rr.ignore = np.zeros((0, 4), np.float32)
        out.append(rr)
    return Split(split.dataset, split.name, list(split.class_names), out)


def by_sequence(split):
    """{sequence id: Split holding that sequence's images}."""
    return {s: Split(split.dataset, split.name, list(split.class_names), [r for r in split.images if r.sequence == s])
            for s in split.sequences}


# --------------------------------------------------------------------------- validation sequences
def sequence_table(split):
    """{sequence id: split_stats + altitude + weather}. Input for pick_val_sequences."""
    return {s: {**split_stats(sp), "altitude": sp.images[0].attrs.get("altitude"), "weather": sp.images[0].attrs.get("weather")}
            for s, sp in by_sequence(split).items()}


def pick_val_sequences(table, alloc, rare=("truck", "bus"), min_rare_seqs=3):
    """Pick validation sequences from `table` (sequence_table of the training pool; never pass test sequences).

    alloc: {(altitude, weather): n sequences}. Among all picks with that allocation in which every `rare` class appears in
    >= min_rare_seqs sequences, return the one whose tiny, truck and bus shares of objects are closest to the whole pool
    (sum of absolute differences). Ties break on the sorted sequence names, so the result is deterministic.
    Returns (sorted sequence ids, distance)."""
    from itertools import combinations, product

    def shares(seqs):
        n = max(sum(table[s]["objects"] for s in seqs), 1)
        return [sum(table[s][k] * table[s]["objects"] for s in seqs) / n for k in ("tiny", *(f"share_{c}" for c in rare))]

    target = shares(list(table))
    cells = [combinations(sorted(s for s, v in table.items() if (v["altitude"], v["weather"]) == c), n) for c, n in alloc.items()]
    best = None
    for pick in product(*cells):
        seqs = sorted(s for group in pick for s in group)
        if any(sum(table[s][f"share_{c}"] > 0 for s in seqs) < min_rare_seqs for c in rare):
            continue
        cand = (sum(abs(a - b) for a, b in zip(shares(seqs), target)), seqs)
        if best is None or cand < best:
            best = cand
    if best is None:
        raise ValueError("no pick satisfies the allocation and the rare-class rule")
    return best[1], best[0]


# --------------------------------------------------------------------------- shared classes
def remap_split(split, mapping, new_names):
    """Re-label GT onto a shared class list. mapping: old class name -> new name, "ignore" (box becomes an
    ignore region: detections on it are neither TP nor FP), or absent/None (box dropped)."""
    import copy
    out = []
    for r in split.images:
        keep, ign = [], [r.ignore]
        for b in r.boxes:
            tgt = mapping.get(split.class_names[int(b[0])])
            if tgt == "ignore":
                ign.append(b[None, 1:5])
            elif tgt is not None:
                nb = b.copy(); nb[0] = new_names.index(tgt); keep.append(nb)
        rr = copy.copy(r)
        rr.boxes = np.array(keep, np.float32).reshape(-1, 7)
        rr.ignore = np.concatenate(ign).astype(np.float32).reshape(-1, 4)
        out.append(rr)
    return Split(split.dataset, split.name, list(new_names), out)


def remap_dets(dets, src_names, mapping, new_names):
    """Same mapping for detections ({key: (N,6)}); classes mapped to None or "ignore" are dropped."""
    lut = np.full(len(src_names), -1)
    for i, n in enumerate(src_names):
        t = mapping.get(n)
        if t not in (None, "ignore"):
            lut[i] = new_names.index(t)
    out = {}
    for k, d in dets.items():
        c = lut[d[:, 5].astype(int)]
        nd = d[c >= 0].copy(); nd[:, 5] = c[c >= 0]
        out[k] = nd
    return out

In [ ]:
%%writefile dronekit_src/dronekit/infer.py
"""Inference engine ("evaluation v2"), raw-detection cache, and merge setups.

Same logic as evaluation(1).ipynb: SAHI's slicer (`get_slice_bboxes`) and SAHI's own merge
functions (`batched_nms`, `batched_greedy_nmm`) called directly with every setting explicit,
so nothing can switch silently. Never call `get_sliced_prediction`.

Pass keys: "full@S" = whole image at imgsz S; "sT@S" = T x T slices, each resized to S.
Detections are (N, 6) float32 arrays: x1, y1, x2, y2, score, class_id (0-based).
Slice passes also return slice boxes (N, 4) so the boundary filter can run later on CPU.

Cache: one parquet per model x split holding every raw pass down to CONF (0.01), plus a
JSON sidecar with timings and settings. Thresholds and merges can change later without the GPU.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

CONF = 0.01           # detections kept down to 0.01
MAX_DET = 1000        # per forward pass
OVERLAP = 0.25        # slice overlap
SLICE_BATCH = 16
EDGE_MARGIN = 2.0     # boundary filter: px from an interior slice edge

# Named setups. "passes" are combined for one model, then merged.
SETUPS = {
    "whole1536":   dict(passes=["full@1536"], method=None),
    "best_sliced": dict(passes=["s512@1280", "full@1536"], method="NMS", metric="IOU", thr=0.6, edge=True),
    "fast":        dict(passes=["s512@1024", "full@1536"], method="NMS", metric="IOU", thr=0.6, edge=True),
}
ENSEMBLE_WBF_THR = 0.65


def parse_pass(key: str):
    kind, sz = key.split("@")
    return ("full", None, int(sz)) if kind == "full" else ("slice", int(kind[1:]), int(sz))


def slice_boxes(H: int, W: int, T: int, overlap: float = OVERLAP):
    from sahi.slicing import get_slice_bboxes
    return get_slice_bboxes(image_height=H, image_width=W, slice_height=T, slice_width=T,
                            auto_slice_resolution=False, overlap_height_ratio=overlap, overlap_width_ratio=overlap)


# --------------------------------------------------------------------------- running a model
def _arr(r, cls_map=None):
    b = r.boxes
    if b is None or len(b) == 0:
        return np.zeros((0, 6), np.float32)
    a = np.c_[b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()].astype(np.float32)
    if cls_map is not None:          # model class index -> dataset class id
        a[:, 5] = np.vectorize(cls_map.__getitem__)(a[:, 5].astype(int))
    return a


def run_pass_on_image(model, img, key, *, conf=CONF, max_det=MAX_DET, half=True, device=0,
                      slice_batch=SLICE_BATCH, overlap=OVERLAP, cls_map=None):
    """One pass on one BGR image. Returns (dets (N,6), slice_boxes (N,4) or None)."""
    kind, T, S = parse_pass(key)
    kw = dict(imgsz=S, conf=conf, max_det=max_det, half=half, device=device, verbose=False)
    if kind == "full":
        return _arr(model.predict(img, **kw)[0], cls_map), None
    H, W = img.shape[:2]
    boxes = slice_boxes(H, W, T, overlap)
    crops = [np.ascontiguousarray(img[y1:y2, x1:x2]) for x1, y1, x2, y2 in boxes]
    D, SB = [], []
    for i in range(0, len(crops), slice_batch):
        rs = model.predict(crops[i:i + slice_batch], **kw)
        for r, (x1, y1, x2, y2) in zip(rs, boxes[i:i + slice_batch]):
            a = _arr(r, cls_map)
            a[:, [0, 2]] += x1
            a[:, [1, 3]] += y1
            D.append(a)
            SB.append(np.tile(np.float32([x1, y1, x2, y2]), (len(a), 1)))
    return np.concatenate(D), np.concatenate(SB).reshape(-1, 4)


COLS = ["image", "pass", "x1", "y1", "x2", "y2", "score", "cls", "sx1", "sy1", "sx2", "sy2"]


def run_and_cache(model, split, passes, out_path, *, meta=None, cuda=True, **kw):
    """Run every pass on every image of `split`; write <out_path>.parquet + <out_path>.json.

    Skips work if the parquet already exists. Returns the cache path.
    Latency = GPU time per image per pass (torch.cuda.synchronize around each image).
    """
    import cv2
    out_path = Path(out_path)
    pq, js = out_path.with_suffix(".parquet"), out_path.with_suffix(".json")
    if pq.exists() and js.exists():
        return pq
    sync = (lambda: __import__("torch").cuda.synchronize()) if cuda else (lambda: None)
    frames, ms = [], {}
    for key in passes:
        t = 0.0
        for rec in split.images:
            img = cv2.imread(str(rec.path))
            sync(); t0 = time.perf_counter()
            d, sb = run_pass_on_image(model, img, key, **kw)
            sync(); t += time.perf_counter() - t0
            if sb is None:
                sb = np.full((len(d), 4), np.nan, np.float32)
            f = pd.DataFrame(np.c_[d, sb].astype(np.float32), columns=COLS[2:])
            f.insert(0, "pass", key); f.insert(0, "image", rec.key)
            frames.append(f)
        ms[key] = 1000 * t / max(len(split), 1)
        print(f"  {out_path.name} {key}: {ms[key]:.0f} ms/img")
    df = pd.concat(frames, ignore_index=True)
    df["image"] = df["image"].astype("category"); df["pass"] = df["pass"].astype("category")
    df["cls"] = df["cls"].astype(np.int16)
    pq.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(pq, compression="zstd", index=False)
    js.write_text(json.dumps(dict(meta or {}, split=split.name, dataset=split.dataset, n_images=len(split),
                                  passes=list(passes), ms_per_img=ms, conf=kw.get("conf", CONF),
                                  max_det=kw.get("max_det", MAX_DET), overlap=kw.get("overlap", OVERLAP),
                                  rows=len(df)), indent=2))
    return pq


def load_cache(path):
    """-> (raw, meta). raw[pass][image_key] = (dets (N,6), slice_boxes (N,4) or None)."""
    path = Path(path)
    df = pd.read_parquet(path.with_suffix(".parquet"))
    meta = json.loads(path.with_suffix(".json").read_text())
    raw = {p: {} for p in meta.get("passes", [])}   # a pass with zero detections still exists
    for (p, im), g in df.groupby(["pass", "image"], observed=True, sort=False):
        a = g[COLS[2:]].to_numpy(np.float32)
        raw.setdefault(str(p), {})[str(im)] = (a[:, :6], None if parse_pass(str(p))[0] == "full" else a[:, 6:])
    return raw, meta


# --------------------------------------------------------------------------- merging
def edge_filter(d, sb, W, H, margin=EDGE_MARGIN):
    """Drop slice detections within `margin` px of an *interior* slice edge (a cut object).
    Edges lying on the image border are not interior and never filter anything."""
    if sb is None or len(d) == 0:
        return d
    x1, y1, x2, y2 = d[:, 0], d[:, 1], d[:, 2], d[:, 3]
    sx1, sy1, sx2, sy2 = sb[:, 0], sb[:, 1], sb[:, 2], sb[:, 3]
    cut = (((sx1 > 0) & (x1 <= sx1 + margin)) | ((sy1 > 0) & (y1 <= sy1 + margin)) |
           ((sx2 < W) & (x2 >= sx2 - margin)) | ((sy2 < H) & (y2 >= sy2 - margin)))
    return d[~cut]


def _overlap(a, b, metric):
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    aa = (a[2] - a[0]) * (a[3] - a[1]); ab = (b[2] - b[0]) * (b[3] - b[1])
    den = min(aa, ab) if metric == "IOS" else (aa + ab - inter)
    return inter / den if den > 0 else 0.0


def merge(parts, method="NMS", metric="IOU", thr=0.5):
    """Class-wise merge with SAHI's own functions."""
    from sahi.postprocess.combine import batched_greedy_nmm, batched_nms
    parts = [p for p in parts if len(p)]
    if not parts:
        return np.zeros((0, 6), np.float32)
    P = np.concatenate(parts)
    if method == "NMS":
        return P[batched_nms(P, match_metric=metric, match_threshold=thr)]
    if method == "GREEDYNMM":                     # same logic as SAHI's _apply_merge: union box, max score
        out = []
        for keep, mlist in batched_greedy_nmm(P, match_metric=metric, match_threshold=thr).items():
            b = P[keep].copy()
            for j in mlist:
                if _overlap(b, P[j], metric) >= thr:
                    b[0], b[1] = min(b[0], P[j, 0]), min(b[1], P[j, 1])
                    b[2], b[3] = max(b[2], P[j, 2]), max(b[3], P[j, 3]); b[4] = max(b[4], P[j, 4])
            out.append(b)
        return np.array(out, np.float32).reshape(-1, 6)
    raise ValueError(method)


def wbf(dets_per_model, W, H, thr=ENSEMBLE_WBF_THR, skip=CONF):
    """Weighted box fusion across models (each model already merged)."""
    from ensemble_boxes import weighted_boxes_fusion
    bl, sl, ll = [], [], []
    for d in dets_per_model:
        bl.append(np.clip(d[:, :4] / [W, H, W, H], 0, 1).tolist()); sl.append(d[:, 4].tolist()); ll.append(d[:, 5].tolist())
    b, s, l = weighted_boxes_fusion(bl, sl, ll, iou_thr=thr, skip_box_thr=skip)
    return np.c_[np.asarray(b).reshape(-1, 4) * [W, H, W, H], s, l].astype(np.float32).reshape(-1, 6)


def apply_setup(raw, split, setup):
    """Merge cached raw passes into final detections. Returns (dets {image_key: (N,6)}, merge ms/img)."""
    if isinstance(setup, str):
        setup = SETUPS[setup]
    passes = setup["passes"]
    missing = [p for p in passes if p not in raw]
    if missing:
        raise KeyError(f"passes not in cache: {missing}")
    out, t0 = {}, time.perf_counter()
    single_full = len(passes) == 1 and parse_pass(passes[0])[0] == "full"
    for rec in split.images:
        parts = []
        for k in passes:
            d, sb = raw[k].get(rec.key, (np.zeros((0, 6), np.float32), None))
            parts.append(edge_filter(d, sb, rec.width, rec.height) if setup.get("edge") else d)
        out[rec.key] = parts[0] if single_full else merge(parts, setup["method"], setup["metric"], setup["thr"])
    return out, 1000 * (time.perf_counter() - t0) / max(len(split), 1)


def ensemble(dets_list, split, thr=ENSEMBLE_WBF_THR):
    """WBF of several models' merged detections, per image."""
    return {r.key: wbf([d[r.key] for d in dets_list], r.width, r.height, thr) for r in split.images}


def setup_ms(meta, setup, merge_ms=0.0):
    """Latency of a setup = sum of its passes' GPU ms/img + merge time (as in eval v2)."""
    if isinstance(setup, str):
        setup = SETUPS[setup]
    return sum(meta["ms_per_img"][p] for p in setup["passes"]) + merge_ms

In [ ]:
%%writefile dronekit_src/dronekit/plots.py
"""Figures: PR curves, normalized confusion matrix, per-class bars, training curves.

Paper figures (white background). Colors follow a validated categorical order; one-series
panels use slot 1, val vs test-dev comparisons use slots 1 and 2 with a legend.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQ_CMAP = ["#ffffff", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"


def _plt():
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
                         "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
                         "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                         "legend.frameon": False, "savefig.dpi": 200, "savefig.bbox": "tight",
                         "figure.facecolor": "white", "axes.facecolor": "white"})
    return plt


def _save(fig, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path)
    import matplotlib.pyplot as plt
    plt.close(fig)
    return path


def plot_pr_curves(curves, class_names, path, title=""):
    """curves: {label: (recall (R,), precision (R, nc))}, e.g. {"val": ..., "test-dev": ...}.
    Small multiples, one panel per class (10 classes would exceed the 8 categorical hues)."""
    plt = _plt()
    nc = len(class_names); ncol = 5; nrow = int(np.ceil(nc / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 2.2 * nrow), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    for k, c in enumerate(class_names):
        ax = axes[k]
        for j, (lbl, (rec, prec)) in enumerate(curves.items()):
            p = prec[:, k]
            ap = np.nanmean(p) if np.isfinite(p).any() else float("nan")
            ax.plot(rec, p, color=SERIES[j], lw=1.6, label=f"{lbl} (AP50 {ap:.2f})")
        ax.set_title(c, fontsize=9); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.legend(fontsize=6.5, loc="lower left")
    for ax in axes[nc:]:
        ax.axis("off")
    for ax in axes[(nrow - 1) * ncol:]:
        ax.set_xlabel("recall")
    for ax in axes[::ncol]:
        ax.set_ylabel("precision")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return _save(fig, path)


def plot_confusion(cm_norm, class_names, path, title="", conf=None):
    """Row-normalized confusion matrix (rows = true class + 'background' FP row)."""
    plt = _plt()
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("seq_blue", SEQ_CMAP)
    labels = list(class_names) + ["background"]
    n = len(labels)
    fig, ax = plt.subplots(figsize=(0.62 * n + 1.6, 0.55 * n + 1.2))
    ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1)
    ax.grid(False)
    for i in range(n):
        for j in range(n):
            v = cm_norm[i, j]
            if v >= 0.005:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.55 else INK)
    ax.set_xticks(range(n), labels, rotation=45, ha="right"); ax.set_yticks(range(n), labels)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    sub = f" (conf ≥ {conf:.2f}, IoU ≥ 0.5)" if conf is not None else ""
    ax.set_title(title + sub, fontsize=9)
    fig.tight_layout()
    return _save(fig, path)


def plot_per_class_bars(df, path, title="", cols=("val", "test-dev")):
    """df: index = class names, columns include `cols` (AP50 values). Grouped horizontal bars."""
    plt = _plt()
    n = len(df); h = 0.38
    fig, ax = plt.subplots(figsize=(6.2, 0.42 * n + 1.0))
    y = np.arange(n)
    for j, c in enumerate(cols):
        if c not in df:
            continue
        ax.barh(y + (j - (len(cols) - 1) / 2) * (h + 0.02), df[c].values, height=h, color=SERIES[j], label=c)
    ax.set_yticks(y, df.index); ax.invert_yaxis()
    ax.set_xlim(0, 1); ax.set_xlabel("AP50"); ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right", ncol=len(cols))
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    return _save(fig, path)


def plot_epoch_scan(df, seq_df, class_names, path, title=""):
    """Per-epoch val AP (IoU 0.50:0.95), three panels on one shared scale: the same detections under three
    scorers; eval-v2 per class; eval-v2 per sequence. df: one row per epoch; seq_df: epoch, sequence, AP."""
    plt = _plt()
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.5), sharex=True, sharey=True)
    scorers = [("ultra_AP", "Ultralytics validator"), ("noign_AP", "eval-v2, ignore regions removed"),
               ("ev2_AP", "eval-v2, ignore regions as crowd")]
    for j, (col, lbl) in enumerate(scorers):
        axes[0].plot(df["epoch"], df[col], color=SERIES[j], lw=2, label=lbl)
        k = df[col].idxmax()
        axes[0].plot(df["epoch"][k], df[col][k], "o", color=SERIES[j], ms=8, mec="white", mew=2, zorder=5)
    axes[0].set_title("Scorer (dot = best epoch)", fontsize=9.5)
    for j, c in enumerate(class_names):
        axes[1].plot(df["epoch"], df[f"AP_{c}"], color=SERIES[j], lw=2, label=c)
    axes[1].set_title("Eval-v2, per class", fontsize=9.5)
    for j, (s, g) in enumerate(seq_df.groupby("sequence")):
        axes[2].plot(g["epoch"], g["AP"], color=SERIES[j], lw=2, label=s)
    axes[2].set_title("Eval-v2, per val sequence", fontsize=9.5)
    axes[0].set_ylabel("AP (IoU 0.50:0.95)")
    for ax in axes:
        ax.set_xlabel("epoch"); ax.set_ylim(bottom=0)
        ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2 if ax is not axes[0] else 1)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    return _save(fig, path)


def plot_training_curves(results_csv, path, title=""):
    """Ultralytics results.csv -> small multiples: each loss and each validator metric in its own panel.
    Note: these metrics come from the Ultralytics validator, not the eval-v2 scorer."""
    import pandas as pd
    plt = _plt()
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    x = df["epoch"] if "epoch" in df else np.arange(1, len(df) + 1)
    cols = [c for c in df.columns if c.startswith(("train/", "val/", "metrics/"))]
    ncol = 4; nrow = int(np.ceil(len(cols) / ncol)) or 1
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.0 * nrow), sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        ax.plot(x, df[c], color=SERIES[0], lw=1.6, marker="o", ms=3)
        ax.set_title(c.replace("metrics/", "").replace("(B)", ""), fontsize=8.5)
    for ax in axes[len(cols):]:
        ax.axis("off")
    for ax in axes[-ncol:]:
        ax.set_xlabel("epoch")
    fig.suptitle(title + "  (validator metrics: Ultralytics, imgsz from training)", fontsize=9.5)
    fig.tight_layout()
    return _save(fig, path)

In [ ]:
%%writefile dronekit_src/dronekit/report.py
"""Training provenance from Ultralytics run files, and small markdown-table helpers."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


def read_results_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def read_run(weights=None, results_csv=None, args_yaml=None):
    """Collect what a training run recorded: checkpoint train_args, args.yaml, results.csv.
    Every value comes from those files; missing ones stay None."""
    info, ta = {}, {}
    if weights is not None and Path(weights).exists():
        try:
            import torch
            ck = torch.load(weights, map_location="cpu", weights_only=False)
            ta = dict(ck.get("train_args") or {})
            info.update(ultralytics_version=ck.get("version"), checkpoint_date=ck.get("date"))
        except Exception as e:                       # keep going; provenance is best-effort per field
            info["checkpoint_error"] = repr(e)[:200]
    if args_yaml is not None and Path(args_yaml).exists():
        import yaml
        ta = {**ta, **(yaml.safe_load(Path(args_yaml).read_text()) or {})}
    for k in ("model", "data", "imgsz", "batch", "optimizer", "lr0", "epochs", "time", "seed", "device"):
        info[f"arg_{k}"] = ta.get(k)
    if results_csv is not None and Path(results_csv).exists():
        df = read_results_csv(results_csv)
        info["epochs_completed"] = int(len(df))
        if "time" in df:
            info["train_hours"] = round(float(df["time"].iloc[-1]) / 3600, 2)
        key = "metrics/mAP50-95(B)"
        if key in df:
            b = int(df[key].idxmax())
            info["best_epoch"] = int(df["epoch"].iloc[b]) if "epoch" in df else b + 1
            info["best_val_mAP50-95_ultralytics"] = round(float(df[key].iloc[b]), 4)
            if "metrics/mAP50(B)" in df:
                info["best_val_mAP50_ultralytics"] = round(float(df["metrics/mAP50(B)"].iloc[b]), 4)
    return info


def md_table(df, floatfmt="{:.3f}"):
    """DataFrame -> GitHub markdown table (no tabulate dependency)."""
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "" if np.isnan(v) else floatfmt.format(v)
        return "" if v is None else str(v)
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(f(r[c]) for c in cols) + " |")
    return "\n".join(lines)

In [ ]:
%%writefile dronekit_src/dronekit/score.py
"""Scorer ("evaluation v2"): COCO AP via pycocotools with the VisDrone settings.

- maxDets = [1, 100, 500]; every number reads the 500 index (`stats` hard-codes 100)
- ignore regions inserted as crowd boxes for every class -> detections inside are neither TP nor FP
- "others" already dropped by the loader
- size buckets: tiny < 16^2, small 16^2-32^2, medium+ >= 32^2 (plus COCO small < 32^2)
Also: per-class AP50, precision/recall at a fixed confidence, PR curves, confusion matrix.
"""
from __future__ import annotations

import contextlib
import io

import numpy as np

AREAS = {"all": [0, 1e10], "tiny": [0, 16 ** 2], "small": [16 ** 2, 32 ** 2],
         "coco_small": [0, 32 ** 2], "medium+": [32 ** 2, 1e10]}
MAX_DETS = [1, 100, 500]


def coco_gt(split) -> dict:
    """COCO-format GT dict. Image id = position in split.images; category id = class_id + 1."""
    images, anns, aid = [], [], 1
    nc = len(split.class_names)
    for i, r in enumerate(split.images):
        images.append(dict(id=i, file_name=r.path.name, width=r.width, height=r.height))
        for x1, y1, x2, y2 in r.ignore:
            w, h = float(x2 - x1), float(y2 - y1)
            for c in range(1, nc + 1):
                anns.append(dict(id=aid, image_id=i, category_id=c, bbox=[float(x1), float(y1), w, h],
                                 area=w * h, iscrowd=1)); aid += 1
        for c, x1, y1, x2, y2, *_ in r.boxes:
            w, h = float(x2 - x1), float(y2 - y1)
            anns.append(dict(id=aid, image_id=i, category_id=int(c) + 1, bbox=[float(x1), float(y1), w, h],
                             area=w * h, iscrowd=0)); aid += 1
    return dict(images=images, annotations=anns,
                categories=[dict(id=k + 1, name=n) for k, n in enumerate(split.class_names)])


def _coco(split):
    from pycocotools.coco import COCO
    with contextlib.redirect_stdout(io.StringIO()):
        G = COCO(); G.dataset = coco_gt(split); G.createIndex()
    return G


def _ap(p):
    p = p[p > -1]
    return float(p.mean()) if p.size else float("nan")


def run_cocoeval(split, dets, gt=None):
    """dets: {image_key: (N,6) x1,y1,x2,y2,score,cls}. Returns the evaluated+accumulated COCOeval, or None."""
    from pycocotools.cocoeval import COCOeval
    G = gt or _coco(split)
    rows = []
    for i, r in enumerate(split.images):
        d = dets.get(r.key)
        if d is not None and len(d):
            rows.append(np.c_[np.full(len(d), i), d[:, 0], d[:, 1], d[:, 2] - d[:, 0], d[:, 3] - d[:, 1],
                              d[:, 4], d[:, 5] + 1])
    if not rows:
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        dt = G.loadRes(np.concatenate(rows).astype(np.float64))
        E = COCOeval(G, dt, "bbox")
        E.params.imgIds = list(range(len(split.images)))
        E.params.maxDets = list(MAX_DETS)
        E.params.areaRng = list(AREAS.values()); E.params.areaRngLbl = list(AREAS.keys())
        E.evaluate(); E.accumulate()
    return E


def metrics_from_eval(E, class_names):
    """-> (metrics dict, per-class AP50 dict). Same keys as evaluation v2."""
    if E is None:
        return {"AP": 0.0, "AP50": 0.0}, {c: 0.0 for c in class_names}
    P = E.eval["precision"]                       # [IoU, recall, class, area, maxDets]
    m = {"AP": _ap(P[:, :, :, 0, -1]), "AP50": _ap(P[0, :, :, 0, -1]), "AP75": _ap(P[5, :, :, 0, -1])}
    for a, lbl in enumerate(AREAS):
        if lbl != "all":
            m[f"AP_{lbl}"] = _ap(P[:, :, :, a, -1]); m[f"AP50_{lbl}"] = _ap(P[0, :, :, a, -1])
    per_cls = {c: _ap(P[0, :, k, 0, -1]) for k, c in enumerate(class_names)}
    return m, per_cls


def evaluate(split, dets, gt=None):
    """-> (metrics, per-class AP50, COCOeval)."""
    E = run_cocoeval(split, dets, gt)
    m, pc = metrics_from_eval(E, split.class_names)
    return m, pc, E


def per_class_ap(E, class_names):
    """Per-class AP (IoU 0.50:0.95, area all, maxDets 500). NaN for a class with no GT."""
    if E is None:
        return {c: 0.0 for c in class_names}
    P = E.eval["precision"]
    return {c: _ap(P[:, :, k, 0, -1]) for k, c in enumerate(class_names)}


def ignore_share(split, dets, conf=0.25):
    """Share of detections with score >= conf lying >= 50% inside an ignore region (same 50% rule as
    confusion_matrix). -> (share, number of detections with score >= conf)."""
    inside = total = 0
    for r in split.images:
        d = dets.get(r.key)
        if d is None or not len(d):
            continue
        d = d[d[:, 4] >= conf]
        total += len(d)
        if len(d) and len(r.ignore):
            inter, _, area = _iou(d[:, :4], r.ignore)
            inside += int(((inter / np.clip(area[:, None], 1e-9, None)).max(1) >= 0.5).sum())
    return inside / total if total else 0.0, total


def matches_at_iou50(E):
    """Per-detection outcome at IoU 0.5, area 'all', maxDets 500, using COCOeval's own matching.

    Returns (scores, is_tp, n_gt) per class: scores/is_tp are arrays of non-ignored detections
    (detections matched to a crowd/ignore box are dropped: neither TP nor FP).
    """
    p = E.params
    nA, nI = len(p.areaRng), len(p.imgIds)
    out = {}
    for k, cat in enumerate(p.catIds):
        sc, tp, ngt = [], [], 0
        for i in range(nI):
            e = E.evalImgs[k * nA * nI + 0 * nI + i]           # area index 0 = 'all'
            if e is None:
                continue
            ngt += int((~np.asarray(e["gtIgnore"], bool)).sum())
            dm, di = e["dtMatches"][0], e["dtIgnore"][0]
            keep = ~np.asarray(di, bool)
            sc.append(np.asarray(e["dtScores"])[keep]); tp.append((np.asarray(dm) > 0)[keep])
        out[cat] = (np.concatenate(sc) if sc else np.zeros(0), np.concatenate(tp) if tp else np.zeros(0, bool), ngt)
    return out


def pr_at_conf(E, conf):
    """Pooled (all classes) precision/recall/F1 at IoU 0.5 for detections with score >= conf."""
    if E is None:
        return dict(precision=0.0, recall=0.0, f1=0.0, tp=0, fp=0, n_gt=0)
    tp = fp = ngt = 0
    for sc, t, n in matches_at_iou50(E).values():
        s = sc >= conf
        tp += int(t[s].sum()); fp += int((~t[s]).sum()); ngt += n
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / ngt if ngt else 0.0
    return dict(precision=p, recall=r, f1=2 * p * r / (p + r) if p + r else 0.0, tp=tp, fp=fp, n_gt=ngt)


def best_f1_conf(E, grid=None):
    """Confidence with the highest pooled F1 (pick on val, reuse unchanged on test)."""
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2) if grid is None else grid
    M = matches_at_iou50(E)
    sc = np.concatenate([v[0] for v in M.values()]); t = np.concatenate([v[1] for v in M.values()])
    ngt = sum(v[2] for v in M.values())
    best = (0.0, float(grid[0]))
    for c in grid:
        s = sc >= c; tp = t[s].sum(); fp = (~t[s]).sum()
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / ngt if ngt else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        if f1 > best[0]:
            best = (f1, float(c))
    return best[1]


def pr_curves(E):
    """-> (recall thresholds (101,), precision (101, n_classes)) at IoU 0.5, area all, maxDets 500.
    COCO's interpolated precision; -1 (no GT) becomes NaN."""
    P = E.eval["precision"][0, :, :, 0, -1].astype(float)
    P[P < 0] = np.nan
    return np.asarray(E.params.recThrs), P


def _iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter, inter / np.clip(aa[:, None] + ab[None, :] - inter, 1e-9, None), aa


def confusion_matrix(split, dets, conf, iou_thr=0.5):
    """(nc+1, nc+1) counts; rows = true class (+ last row 'background' = false positives),
    columns = predicted class (+ last column 'background' = missed objects).

    Class-agnostic greedy matching by score at IoU >= iou_thr. Unmatched detections whose area
    is >= 50% inside an ignore region are dropped, mirroring the crowd rule (COCO uses
    intersection / detection area for crowd boxes).
    """
    nc = len(split.class_names)
    cm = np.zeros((nc + 1, nc + 1), np.int64)
    for r in split.images:
        d = dets.get(r.key, np.zeros((0, 6), np.float32))
        d = d[d[:, 4] >= conf]
        d = d[np.argsort(-d[:, 4], kind="stable")]
        g = r.boxes
        used = np.zeros(len(g), bool)
        unmatched = []
        if len(d) and len(g):
            _, iou, _ = _iou(d[:, :4], g[:, 1:5])
        for j in range(len(d)):
            if len(g):
                cand = np.where(~used & (iou[j] >= iou_thr))[0]
                if len(cand):
                    gi = cand[np.argmax(iou[j, cand])]
                    used[gi] = True
                    cm[int(g[gi, 0]), int(d[j, 5])] += 1
                    continue
            unmatched.append(j)
        if unmatched:
            u = d[unmatched]
            if len(r.ignore):
                inter, _, area = _iou(u[:, :4], r.ignore)
                u = u[(inter / np.clip(area[:, None], 1e-9, None)).max(1) < 0.5]
            for c in u[:, 5].astype(int):
                cm[nc, c] += 1
        for gi in np.where(~used)[0]:
            cm[int(g[gi, 0]), nc] += 1
    return cm


def normalize_rows(cm):
    s = cm.sum(1, keepdims=True).astype(float)
    return np.divide(cm, s, out=np.zeros_like(cm, float), where=s > 0)

In [ ]:
%%writefile dronekit_src/dronekit/tiles.py
"""Training-set builder: the YOLO26m-Tile "D2" recipe for any dataset in the common format.

Per image: ignore regions painted gray (114) -> saved as a full image with YOLO labels; then 640 and 960 px
tiles with 25% overlap (step = T * 0.75); a cut box is kept if >= 60% of it lies inside the tile, otherwise
painted gray; empty tiles kept with probability 0.1, seeded per image (random.Random(image key)).
Same arithmetic as the visdrone-training notebook (Cell 3), including the 6-decimal YOLO label round-trip,
so VisDrone input reproduces the original 6,471 + 63,745 images.
"""
from __future__ import annotations

import os
import random
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np

TILE_SIZES = (640, 960)
OVERLAP = 0.25
MIN_VIS = 0.6
EMPTY_KEEP = 0.1
GREY = 114


def starts(L, T, step):
    if L <= T:
        return [0]
    s = list(range(0, L - T + 1, step))
    if s[-1] + T < L:
        s.append(L - T)
    return s


def yolo_lines(boxes, W, H):
    """Clip to the image, drop empty boxes, format as YOLO lines (class cx cy w h, 6 decimals)."""
    out = []
    for c, x1, y1, x2, y2, *_ in boxes.tolist():      # Python floats: same rounding as the original notebook
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        out.append(f"{int(c)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
    return out


def _boxes_from_lines(lines, W, H):
    out = []
    for r in lines:
        c, cx, cy, w, h = r.split()
        cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
        out.append([int(c), cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
    return np.array(out, dtype=np.float32).reshape(-1, 5)


def _process(args):
    import cv2
    rec, out_full, out_tiles, tile_sizes, overlap, min_vis, empty_keep, grey = args
    rng = random.Random(rec.key)
    img = cv2.imread(str(rec.path))
    H, W = img.shape[:2]
    for x1, y1, x2, y2 in rec.ignore.astype(int):
        x, y, w, h = x1, y1, x2 - x1, y2 - y1
        img[max(0, y):y + h, max(0, x):x + w] = grey
    lines = yolo_lines(rec.boxes, W, H)
    cv2.imwrite(str(out_full / "images" / f"{rec.key}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    (out_full / "labels" / f"{rec.key}.txt").write_text("\n".join(lines))
    boxes = _boxes_from_lines(lines, W, H)
    n = 0
    for T in tile_sizes:
        if T >= max(W, H):
            continue
        step = int(T * (1 - overlap))
        for y0 in starts(H, T, step):
            for x0 in starts(W, T, step):
                x1, y1, x2, y2 = x0, y0, min(x0 + T, W), min(y0 + T, H)
                tile = img[y1:y2, x1:x2].copy()
                tw, th = x2 - x1, y2 - y1
                tl = []
                if len(boxes):
                    bx1 = np.clip(boxes[:, 1], x1, x2); by1 = np.clip(boxes[:, 2], y1, y2)
                    bx2 = np.clip(boxes[:, 3], x1, x2); by2 = np.clip(boxes[:, 4], y1, y2)
                    inter = (bx2 - bx1) * (by2 - by1)
                    area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 4] - boxes[:, 2])
                    vis = np.where(area > 0, inter / area, 0)
                    for i in np.where(inter > 0)[0]:
                        if vis[i] >= min_vis:
                            tl.append(f"{int(boxes[i, 0])} {((bx1[i] + bx2[i]) / 2 - x1) / tw:.6f} "
                                      f"{((by1[i] + by2[i]) / 2 - y1) / th:.6f} {(bx2[i] - bx1[i]) / tw:.6f} "
                                      f"{(by2[i] - by1[i]) / th:.6f}")
                        else:   # gray out badly-cut objects so they aren't unlabeled positives
                            tile[int(by1[i] - y1):int(by2[i] - y1), int(bx1[i] - x1):int(bx2[i] - x1)] = grey
                if not tl and rng.random() > empty_keep:
                    continue
                name = f"{rec.key}_t{T}_{x0}_{y0}"
                cv2.imwrite(str(out_tiles / "images" / f"{name}.jpg"), tile, [cv2.IMWRITE_JPEG_QUALITY, 95])
                (out_tiles / "labels" / f"{name}.txt").write_text("\n".join(tl))
                n += 1
    return n


def build_d2(split, out_dir, tile_sizes=TILE_SIZES, overlap=OVERLAP, min_vis=MIN_VIS, empty_keep=EMPTY_KEEP,
             grey=GREY, workers=None):
    """Write <out_dir>/full/{images,labels} and <out_dir>/tiles/{images,labels}. Returns (n_full, n_tiles)."""
    out_dir = Path(out_dir)
    full, tiles = out_dir / "full", out_dir / "tiles"
    for d in (full, tiles):
        (d / "images").mkdir(parents=True, exist_ok=True); (d / "labels").mkdir(parents=True, exist_ok=True)
    jobs = [(r, full, tiles, tuple(tile_sizes), overlap, min_vis, empty_keep, grey) for r in split.images]
    if workers == 0:
        counts = [_process(j) for j in jobs]
    else:
        with ProcessPoolExecutor(max_workers=workers or os.cpu_count()) as ex:
            counts = list(ex.map(_process, jobs, chunksize=16))
    return len(jobs), int(sum(counts))


# --------------------------------------------------------------------------- YOLO26s recipe (teammate's prompt1/prompt2 notebooks)
SF_TILE = 640
SF_OVERLAP = 0.20
SF_MIN_VIS = 0.30
SF_MIN_SIDE = 2
SF_EMPTY_KEEP = 0.10
SF_SEED = 42


def sf_tile_plan(rec, W, H):
    """Kept tiles of one image under the YOLO26s recipe: [(x1, y1, x2, y2, label lines)]. Labels only, so it can be counted without images.
    A box stays if >= 30% of it is inside the tile and both clipped sides are >= 2 px; other cut boxes are dropped, not painted.
    Empty tiles are kept with probability 0.1, seeded per image."""
    from sahi.slicing import get_slice_bboxes
    rng = random.Random(f"{SF_SEED}-{rec.key}")
    b = rec.boxes
    area = (b[:, 3] - b[:, 1]) * (b[:, 4] - b[:, 2])
    out = []
    for x1, y1, x2, y2 in get_slice_bboxes(image_height=H, image_width=W, slice_height=SF_TILE, slice_width=SF_TILE, auto_slice_resolution=False,
                                           overlap_height_ratio=SF_OVERLAP, overlap_width_ratio=SF_OVERLAP):
        cx1, cy1 = np.clip(b[:, 1], x1, x2), np.clip(b[:, 2], y1, y2)
        cx2, cy2 = np.clip(b[:, 3], x1, x2), np.clip(b[:, 4], y1, y2)
        cw, ch = cx2 - cx1, cy2 - cy1
        keep = (cw >= SF_MIN_SIDE) & (ch >= SF_MIN_SIDE) & (cw * ch >= SF_MIN_VIS * area)
        if not keep.any() and rng.random() > SF_EMPTY_KEEP:
            continue
        tw, th = x2 - x1, y2 - y1
        lines = [f"{int(b[i, 0])} {(cx1[i] - x1 + cw[i] / 2) / tw:.6f} {(cy1[i] - y1 + ch[i] / 2) / th:.6f} {cw[i] / tw:.6f} {ch[i] / th:.6f}"
                 for i in np.where(keep)[0]]
        out.append((x1, y1, x2, y2, lines))
    return out


def _process_sf(args):
    import cv2
    rec, out_full, out_tiles, grey = args
    img = cv2.imread(str(rec.path))
    H, W = img.shape[:2]
    for x1, y1, x2, y2 in rec.ignore.astype(int):
        img[max(0, y1):y2, max(0, x1):x2] = grey
    cv2.imwrite(str(out_full / "images" / f"{rec.key}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    (out_full / "labels" / f"{rec.key}.txt").write_text("\n".join(yolo_lines(rec.boxes, W, H)))
    plan = sf_tile_plan(rec, W, H)
    for x1, y1, x2, y2, lines in plan:
        name = f"{rec.key}_{x1}_{y1}"
        cv2.imwrite(str(out_tiles / "images" / f"{name}.jpg"), img[y1:y2, x1:x2], [cv2.IMWRITE_JPEG_QUALITY, 95])
        (out_tiles / "labels" / f"{name}.txt").write_text("\n".join(lines))
    return len(plan)


def build_sf(split, out_dir, grey=GREY, workers=None):
    """YOLO26s training set: <out_dir>/full and <out_dir>/tiles, each with images/ and labels/. Returns (n_full, n_tiles)."""
    out_dir = Path(out_dir)
    full, tiles = out_dir / "full", out_dir / "tiles"
    for d in (full, tiles):
        (d / "images").mkdir(parents=True, exist_ok=True); (d / "labels").mkdir(parents=True, exist_ok=True)
    jobs = [(r, full, tiles, grey) for r in split.images]
    if workers == 0:
        counts = [_process_sf(j) for j in jobs]
    else:
        with ProcessPoolExecutor(max_workers=workers or os.cpu_count()) as ex:
            counts = list(ex.map(_process_sf, jobs, chunksize=16))
    return len(jobs), int(sum(counts))


def write_yaml(path, root, train_dirs, val_dir, names):
    """Ultralytics dataset yaml. train_dirs/val_dir are relative to root."""
    tr = "\n".join(f"  - {d}" for d in train_dirs)
    nb = "\n".join(f"  {i}: {n}" for i, n in enumerate(names))
    Path(path).write_text(f"path: {root}\ntrain:\n{tr}\nval: {val_dir}\nnames:\n{nb}\n")
    return Path(path)

In [ ]:
%%writefile dronekit_src/dronekit/vlm.py
"""Prompts and strict parsing for the VLM counting systems. Model-free, so all of it is unit tested on CPU.

B direct: integer answer. C localize-then-count: one point per object, count = valid distinct points.
D grounded: the detector's boxes drawn on the image and listed; the VLM gives the final count. Unparsable answers count as wrong.
"""
from __future__ import annotations

import re

NOUNS = {"pedestrian": "pedestrians (people standing or walking)", "people": "people who are sitting or riding, not standing or walking",
         "bicycle": "bicycles", "car": "cars", "van": "vans", "truck": "trucks", "tricycle": "tricycles", "awning-tricycle": "tricycles with an awning",
         "bus": "buses", "motor": "motorcycles"}
MAX_POINT = 1000
MAX_LISTED = 150      # boxes listed in the prompt, highest scores first; all are drawn on the image


def prompt_direct(target):
    return f"Count the {NOUNS[target]} in this aerial image. Answer with one integer and nothing else."


def prompt_localize(target):
    return (f"List every one of the {NOUNS[target]} in this aerial image. Write one line per object as x,y: the center of the object, where x and y are "
            f"integers from 0 to {MAX_POINT} relative to the image width and height. Write only those lines. If there are none, write NONE.")


def prompt_grounded(target, boxes):
    lines = "\n".join(f"{i + 1}: {x1},{y1},{x2},{y2} score {s:.2f}" for i, (x1, y1, x2, y2, s) in enumerate(boxes)) or "(none)"
    return (f"An object detector marked {len(boxes)} candidate {NOUNS[target]} in this aerial image. They are drawn as numbered green boxes and listed below "
            f"as number: x1,y1,x2,y2 in pixels. The detector may have false positives and misses. Decide the true number of {NOUNS[target]}. "
            f"Answer with one integer and nothing else.\n{lines}")


def parse_int(text):
    """A bare integer, optionally with a final period. Anything else is unparsable (None)."""
    m = re.fullmatch(r"\s*(\d{1,6})\s*\.?\s*", text or "")
    return int(m[1]) if m else None


def parse_points(text):
    """-> count of distinct valid points, or None when the answer has no valid line and is not NONE.
    A final partial line (cut by the token limit) is ignored because it fails the line pattern."""
    t = (text or "").strip()
    if re.fullmatch(r"(?i)none\.?", t):
        return 0
    pts = set()
    for line in t.splitlines():
        m = re.fullmatch(r"\s*\(?\s*(\d{1,4})\s*,\s*(\d{1,4})\s*\)?\s*", line)
        if m and int(m[1]) <= MAX_POINT and int(m[2]) <= MAX_POINT:
            pts.add((int(m[1]), int(m[2])))
    return len(pts) if pts else None

In [ ]:
%%writefile dronekit_src/tests/conftest.py
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from dronekit.data import ImageRecord, Split  # noqa: E402


def make_split(images, class_names=("a", "b")):
    """images: list of dict(boxes=[[cls,x1,y1,x2,y2],...], ignore=[[x1,y1,x2,y2],...], wh=(W,H))."""
    recs = []
    for i, im in enumerate(images):
        b = np.array([list(r) + [0, 0] for r in im.get("boxes", [])], np.float32).reshape(-1, 7)
        g = np.array(im.get("ignore", []), np.float32).reshape(-1, 4)
        W, H = im.get("wh", (1000, 1000))
        recs.append(ImageRecord(f"img{i:03d}", Path(f"img{i:03d}.jpg"), W, H, "seq0", b, g))
    return Split("toy", "toy", list(class_names), recs)


@pytest.fixture
def split_factory():
    return make_split

In [ ]:
%%writefile dronekit_src/tests/test_conformal.py
import numpy as np

from dronekit import conformal as C


def test_quantile_rank_and_small_n():
    assert C.conformal_quantile(np.arange(1, 10), 0.1) == 9          # ceil(10 * 0.9) = 9th of 9
    assert C.conformal_quantile(np.arange(1, 9), 0.1) == np.inf      # n = 8: rank 9 does not exist


def test_interval_is_exactly_the_scores_within_q():
    for kind in C.SCORES:
        for yhat in (0, 3, 40):
            lo, hi = C.interval(np.array([yhat]), 1.3, kind)
            for y in range(0, 80):
                assert (lo[0] <= y <= hi[0]) == (C.score(y, yhat, kind) <= 1.3 + 1e-9)


def test_image_level_coverage_on_exchangeable_data():
    # target met within sampling error on iid data, over many calibration/test splits
    rng = np.random.default_rng(0)
    for alpha in (0.1, 0.2):
        cov = []
        for _ in range(300):
            lam = rng.uniform(1, 40, 700)
            y, yhat = rng.poisson(lam), rng.poisson(lam)
            s = C.score(y, yhat)
            q = C.calibrate(s[:200], alpha=alpha, method="image")
            lo, hi = C.interval(yhat[200:], q)
            cov.append(C.coverage(y[200:], lo, hi))
        assert abs(np.mean(cov) - (1 - alpha)) < 0.015
        assert np.mean(cov) >= 1 - alpha - 0.005                     # marginally valid, integer scores only make it wider


def _seq_data(rng, n_seq, n_frames=30, sd=1.0):
    # each sequence has its own bias: frames of a sequence are correlated, sequences are exchangeable
    y, yhat, g = [], [], []
    for i in range(n_seq):
        b = rng.normal(0, sd)
        lam = rng.uniform(5, 40, n_frames)
        t = rng.poisson(lam)
        y.append(t); yhat.append(np.maximum(np.round(t * np.exp(b * 0.3) + rng.normal(0, 1, n_frames)), 0)); g.append(np.full(n_frames, i))
    return np.concatenate(y), np.concatenate(yhat), np.concatenate(g)


def test_sequence_calibration_guarantee_and_image_baseline_undercovers():
    # sequence guarantee: with probability >= 1 - alpha a new sequence has >= 1 - alpha of its frames covered
    rng = np.random.default_rng(1)
    alpha, hit, img_cov, seq_cov = 0.1, [], [], []
    for _ in range(200):
        y, yhat, g = _seq_data(rng, 30)
        tst = g >= 19                                                # 19 calibration sequences are the fewest that allow alpha 0.1
        s = C.score(y, yhat)
        q = C.calibrate(s[~tst], g[~tst], alpha)
        lo, hi = C.interval(yhat[tst], q)
        hit += [v >= 1 - alpha for v in C.sequence_coverage(y[tst], lo, hi, g[tst]).values()]
        seq_cov.append(C.coverage(y[tst], lo, hi))
        qi = C.calibrate(s[~tst], alpha=alpha, method="image")
        lo, hi = C.interval(yhat[tst], qi)
        img_cov.append(np.mean([v >= 1 - alpha for v in C.sequence_coverage(y[tst], lo, hi, g[tst]).values()]))
    assert np.mean(hit) >= 1 - alpha - 0.02
    assert np.mean(img_cov) < np.mean(hit)                           # frame-level calibration does not hold per sequence


def test_mondrian_covers_each_stratum():
    rng = np.random.default_rng(2)
    n = 6000
    k = rng.integers(0, 2, n)
    y = rng.poisson(10, n)
    yhat = np.maximum(y + np.round(rng.normal(0, np.where(k == 1, 6, 1))), 0)
    s = C.score(y, yhat, "abs")
    q = C.mondrian(s[:3000], k[:3000], alpha=0.1, method="image")
    assert q[1] > q[0]
    for j in (0, 1):
        m = k[3000:] == j
        lo, hi = C.interval(yhat[3000:][m], q[j], "abs")
        assert C.coverage(y[3000:][m], lo, hi) >= 0.87              # stratum 0's errors are nearly all 0 or 1, so it covers well above target


def test_pooled_targets_frame_coverage_of_a_new_sequence():
    rng = np.random.default_rng(3)
    pooled, seq, width_p, width_s = [], [], [], []
    for _ in range(200):
        y, yhat, g = _seq_data(rng, 30)
        tst = g >= 19
        s = C.score(y, yhat)
        for m, cov, wid in (("pooled", pooled, width_p), ("sequence", seq, width_s)):
            lo, hi = C.interval(yhat[tst], C.calibrate(s[~tst], g[~tst], 0.1, m))
            cov.append(C.coverage(y[tst], lo, hi)); wid.append(C.mean_width(lo, hi))
    assert np.mean(pooled) >= 0.85 and np.mean(seq) >= np.mean(pooled) and np.mean(width_s) >= np.mean(width_p)
    assert C.pooled_quantile([1, 2, 3], [0, 0, 0], 0.1) == np.inf    # one sequence: +inf holds half the weight


def test_few_shot_draws_use_whole_sequences_first():
    draws = C.few_shot_draws({"a": 4, "b": 10, "c": 3}, 5, n_draws=30)
    for d in draws:
        assert sum(n for _, n in d) == 5 and len({s for s, _ in d}) == len(d)
    assert len({tuple(d) for d in draws}) > 1


def test_baselines_and_bootstrap():
    lo, hi = C.poisson_interval(np.array([0.0, 10.0]))
    assert lo[1] <= 10 <= hi[1] and hi[0] == 0
    lo, hi = C.residual_range(np.array([10.0]), [12, 8, 10, 9, 11], [10, 10, 10, 10, 10])
    assert lo[0] <= 10 <= hi[0]
    a, b = C.bootstrap_ci([1, 1, 3, 3], [0, 0, 1, 1], n=200)
    assert 1 <= a <= b <= 3

In [ ]:
%%writefile dronekit_src/tests/test_countset.py
import numpy as np
import pandas as pd

from dronekit import countset as K
from dronekit.data import ImageRecord, Split


def _split():
    # 100 x 100. Classes: pedestrian, people, car. Object D sits in the ignore region; object E on the x = 50 border.
    b = [[0, 10, 10, 20, 20, 0, 0],    # A pedestrian, top left
         [1, 70, 10, 80, 20, 1, 1],    # B people, top right
         [2, 10, 70, 30, 90, 2, 0],    # C car, bottom left
         [2, 2, 2, 8, 8, 0, 0],        # D car, centered inside ignore
         [2, 45, 40, 55, 60, 0, 0]]    # E car, center exactly on x = 50, y = 50
    r = ImageRecord("im", "im.jpg", 100, 100, "s1", np.float32(b), np.float32([[0, 0, 10, 10]]))
    return Split("visdrone", "val", ["pedestrian", "people", "car"], [r])


def _get(rows, target, region):
    return next(x for x in rows if x["target"] == target and x["region"] == region)


def test_counts_centers_and_ignore():
    rows = K.records(_split())
    assert _get(rows, "car", "whole")["count"] == 2                # D excluded by its center
    assert _get(rows, "people", "whole")["count"] == 2             # group: pedestrian + people
    assert _get(rows, "pedestrian", "left")["count"] == 1
    assert _get(rows, "car", "left")["count"] == 1 and _get(rows, "car", "right")["count"] == 1   # E counted once, on the right
    assert sum(_get(rows, "car", q)["count"] for q in ("q1", "q2", "q3", "q4")) == 2


def test_stratification_fields():
    rows = K.records(_split())
    r = _get(rows, "people", "whole")
    assert r["count_bin"] == "1-8" and r["median_side"] == 10 and np.isclose(r["median_area_frac"], 0.01)
    assert np.isclose(r["density"], 60 / 10)                       # the two centers are 60 px apart, sides 10
    assert r["frac_occluded"] == 0.5 and r["frac_truncated"] == 0.5     # B is occluded and truncated, A is neither
    assert np.isclose(_get(rows, "car", "whole")["frac_occluded"], 1 / 2)
    e = _get(rows, "pedestrian", "right")
    assert e["count"] == 0 and e["count_bin"] == "0" and np.isnan(e["median_side"]) and np.isnan(e["density"])


def test_heavily_ignored_region_is_dropped():
    sp = _split()
    sp.images[0].ignore = np.float32([[0, 0, 50, 50]])             # all of q1 and 25% of the image
    regions = {x["region"] for x in K.records(sp)}
    assert "q1" not in regions and "left" not in regions and "whole" not in regions and "q2" in regions
    sp.images[0].ignore = np.float32([[0, 0, 10, 10]])             # 1% of the image
    assert "whole" in {x["region"] for x in K.records(sp)}


def test_groups_need_two_classes():
    assert set(K.targets(["car", "truck", "bus"])) == {"car", "truck", "bus", "vehicles"}
    assert set(K.targets(["car"])) == {"car"}


def test_bins():
    assert [K.count_bin(n) for n in (0, 1, 8, 9, 20, 21, 50, 51)] == ["0", "1-8", "1-8", "9-20", "9-20", "21-50", "21-50", ">50"]


def test_tertiles_frozen_on_dev():
    e = K.tertile_edges([1, 2, 3, 4, 5, 6, np.nan])
    assert list(K.tertile([0, 3.5, 99, np.nan], e)) == ["low", "mid", "high", "none"]


def test_count_dets_same_rules_as_gt():
    r = _split().images[0]
    # (x1, y1, x2, y2, score, cls): in ignore; left, kept; left, low score; right; other class
    d = np.float32([[2, 2, 8, 8, .9, 2], [10, 70, 30, 90, .9, 2], [10, 60, 30, 80, .1, 2], [60, 60, 70, 70, .9, 2], [10, 10, 20, 20, .9, 0]])
    assert K.count_dets(d, (100, 100), r.ignore, [2], "whole", {2: .5}) == 2
    assert K.count_dets(d, (100, 100), r.ignore, [2], "left", {2: .5}) == 1
    assert K.count_dets(d, (100, 100), r.ignore, [2], "left", {2: .05}) == 2
    assert K.count_dets(d, (100, 100), r.ignore, [0, 2], "whole", {2: .5}) == 2     # class 0 has no threshold: not counted
    assert K.count_dets(np.zeros((0, 6)), (100, 100), r.ignore, [2]) == 0


def test_subset_is_fixed_balanced_and_unique():
    rng = np.random.default_rng(0)
    rows = [dict(dataset=d, image=f"{d}{i}", region="whole", count=int(rng.integers(1, 80)), count_bin="x", size_bin=rng.choice(["low", "high"]), target="car")
            for d in ("a", "b") for i in range(200)]
    df = pd.DataFrame(rows)
    df["count_bin"] = [K.count_bin(n) for n in df["count"]]
    s = K.stratified_subset(df, 60)
    assert s == K.stratified_subset(df, 60) and len(s) == len(set(s)) == 60
    n = pd.Series([d for d, _ in s]).value_counts()
    assert abs(n["a"] - n["b"]) <= 6                               # filled cell by cell, so datasets differ by at most a few rounds


def test_tune_thresholds_picks_lowest_mae():
    sp = _split()
    # car: 2 real objects; one extra detection at 0.2 makes low thresholds overcount
    d = {"im": np.float32([[10, 70, 30, 90, .9, 2], [45, 40, 55, 60, .8, 2], [60, 60, 70, 70, .2, 2], [2, 2, 8, 8, .95, 2]])}
    assert K.tune_thresholds(sp, d, [2])[2] == 0.21                # first grid point above 0.2; the box in ignore never counts


def test_paired_records_use_same_rules_and_specs():
    sp = _split()
    d = {"im": np.float32([[10, 70, 30, 90, .9, 2], [60, 60, 70, 70, .2, 2], [2, 2, 8, 8, .95, 2], [10, 10, 20, 20, .9, 0]])}
    rows = K.paired_records(sp, d, {2: .5}, {"car": [2]}, {"car": [2]}, ("whole", "left"))
    w = next(r for r in rows if r["region"] == "whole")
    assert (w["y"], w["yhat"]) == (2, 1) and np.isclose(w["pred_side"], 20) and np.isnan(w["pred_density"])
    assert next(r for r in rows if r["region"] == "left")["yhat"] == 1

In [ ]:
%%writefile dronekit_src/tests/test_data.py
import numpy as np
import pytest

from dronekit import data


def test_parse_visdrone_rules():
    txt = "\n".join([
        "10,20,30,40,1,4,0,1",     # car (cat 4 -> class 3), trunc 0, occ 1
        "0,0,50,50,0,1,0,0",       # score 0 -> ignore region
        "5,5,10,10,1,0,0,0",       # category 0 -> ignore region
        "1,1,5,5,1,11,0,0",        # others -> dropped
        "1,1,0,5,1,3,0,0",         # zero width -> dropped
        "7,8,2,3,1,10,1,2,",       # trailing comma; motor (class 9)
    ])
    boxes, ign = data.parse_visdrone_txt(txt)
    assert boxes.tolist() == [[3, 10, 20, 40, 60, 1, 0], [9, 7, 8, 9, 11, 2, 1]]
    assert ign.tolist() == [[0, 0, 50, 50], [5, 5, 15, 15]]


def test_sequence_id():
    assert data.visdrone_sequence("0000001_02999_d_0000005") == "0000001"
    assert data.visdrone_sequence("9999938_00000_d_0000207") == "9999938"


def test_split_stats(split_factory):
    sp = split_factory([dict(boxes=[[0, 0, 0, 10, 10], [1, 0, 0, 40, 40]])])
    st = data.split_stats(sp)
    assert st["objects"] == 2 and np.isclose(st["tiny"], 0.5) and np.isclose(st["medium_plus"], 0.5)


def _row(alt, wx, objects, tiny, truck, bus):
    return dict(altitude=alt, weather=wx, objects=objects, tiny=tiny, share_truck=truck, share_bus=bus)


def test_pick_val_sequences_follows_alloc_and_rare_rule():
    table = dict(a=_row("low", "day", 100, 0.2, 0.0, 0.0), b=_row("low", "day", 100, 0.2, 0.1, 0.1),
                 c=_row("med", "fog", 100, 0.2, 0.1, 0.1), d=_row("med", "fog", 100, 0.2, 0.0, 0.0),
                 e=_row("high", "day", 100, 0.9, 0.5, 0.5))
    # e is outside the allocation; a and d carry no rare class, so only b + c meets the rule
    seqs, _ = data.pick_val_sequences(table, {("low", "day"): 1, ("med", "fog"): 1}, min_rare_seqs=2)
    assert seqs == ["b", "c"]
    with pytest.raises(ValueError):
        data.pick_val_sequences(table, {("low", "day"): 1, ("med", "fog"): 1}, min_rare_seqs=3)


def test_pick_val_sequences_closest_to_pool_and_deterministic():
    table = dict(a=_row("low", "day", 100, 0.5, 0.1, 0.1), b=_row("low", "day", 100, 0.1, 0.1, 0.1),
                 c=_row("low", "day", 100, 0.1, 0.1, 0.1))     # pool tiny share = 0.233: b and c tie, b wins by name
    assert data.pick_val_sequences(table, {("low", "day"): 1}, min_rare_seqs=1)[0] == ["b"]

In [ ]:
%%writefile dronekit_src/tests/test_datasets.py
import numpy as np
import pytest

from dronekit import data, tiles


def _uavdt_fixture(tmp_path):
    gt = tmp_path / "GT"; gt.mkdir()
    # frame, id, x, y, w, h, out-of-view, occlusion, category
    (gt / "M0101_gt_whole.txt").write_text("1,1,10,20,30,40,1,1,1\n1,2,100,100,50,50,3,2,3\n2,1,12,20,30,40,1,4,2\n3,1,14,20,30,40,1,1,1\n")
    (gt / "M0101_gt_ignore.txt").write_text("1,9,500,300,40,20,1,-1,-1\n")
    (tmp_path / "M_attr/train").mkdir(parents=True); (tmp_path / "M_attr/test").mkdir(parents=True)
    (tmp_path / "M_attr/train/M0101_attr.txt").write_text("0,1,0,0,0,1,0,1,0,0\n")
    (tmp_path / "M_attr/test/M0701 _attr.txt").write_text("0,0,1,0,0,1,1,0,0,0\n")   # stray space, as released
    return gt


def test_uavdt_loader(tmp_path):
    gt = _uavdt_fixture(tmp_path)
    attrs, split = data.read_uavdt_attrs(tmp_path / "M_attr")
    assert split == {"M0101": "train", "M0701": "test"}
    assert attrs["M0101"]["weather"] == "night" and attrs["M0101"]["altitude"] == "high-alt"
    sp = data.load_uavdt(tmp_path / "imgs", gt, tmp_path / "M_attr", ["M0101"], stride=1, size=(1080, 540))
    assert [r.key for r in sp.images] == ["M0101_img000001", "M0101_img000002", "M0101_img000003"]
    r = sp.images[0]
    assert r.boxes.tolist() == [[0, 10, 20, 40, 60, 0, 0], [2, 100, 100, 150, 150, 3, 1]]   # occ large=3, out small=1
    assert r.ignore.tolist() == [[500, 300, 540, 320]] and r.sequence == "M0101" and r.attrs["altitude"] == "high-alt"
    assert sp.images[1].boxes[0, 0] == 1 and sp.images[1].boxes[0, 5] == 1                   # truck, small occlusion
    sp2 = data.load_uavdt(tmp_path / "imgs", gt, tmp_path / "M_attr", ["M0101"], stride=2, size=(1080, 540))
    assert [r.key for r in sp2.images] == ["M0101_img000001", "M0101_img000003"]
    # image files with no gt_whole row (frames 4, 5) are unlabeled and must not enter the split as empty frames
    (tmp_path / "imgs/M0101").mkdir(parents=True)
    for f in range(1, 6):
        (tmp_path / f"imgs/M0101/img{f:06d}.jpg").write_bytes(b"")
    sp3 = data.load_uavdt(tmp_path / "imgs", gt, tmp_path / "M_attr", ["M0101"], stride=1, size=(1080, 540))
    assert [r.key for r in sp3.images] == ["M0101_img000001", "M0101_img000002", "M0101_img000003"]


def test_remap_van_to_car_and_van_ignored(split_factory):
    names = ["car", "van", "pedestrian"]
    sp = split_factory([dict(boxes=[[0, 0, 0, 10, 10], [1, 20, 20, 30, 30], [2, 40, 40, 50, 50]])], class_names=names)
    shared = ["car", "truck", "bus"]
    a = data.remap_split(sp, {"car": "car", "van": "car"}, shared)
    assert a.images[0].boxes[:, 0].tolist() == [0, 0] and len(a.images[0].ignore) == 0     # pedestrian dropped
    b = data.remap_split(sp, {"car": "car", "van": "ignore"}, shared)
    assert b.images[0].boxes[:, 0].tolist() == [0] and b.images[0].ignore.tolist() == [[20, 20, 30, 30]]
    d = {"img000": np.float32([[0, 0, 10, 10, .9, 0], [20, 20, 30, 30, .8, 1], [40, 40, 50, 50, .7, 2]])}
    assert data.remap_dets(d, names, {"car": "car", "van": "car"}, shared)["img000"][:, 5].tolist() == [0, 0]
    assert data.remap_dets(d, names, {"car": "car", "van": "ignore"}, shared)["img000"][:, 5].tolist() == [0]


def test_tile_starts_cover_the_image():
    assert tiles.starts(540, 640, 480) == [0]
    assert tiles.starts(1080, 640, 480) == [0, 440]
    assert tiles.starts(1080, 960, 720) == [0, 120]


def test_sf_tile_rules(split_factory):
    # 1080 x 540 -> tiles x 0-640 and x 440-1080. Box A: 10/30 of its width in the first tile (33% >= 30%: kept, clipped).
    # Box B: 5/30 in the first tile (17%: dropped there). Box C: 1 px wide after clipping (below 2 px: dropped there).
    sp = split_factory([dict(boxes=[[0, 630, 100, 660, 130], [1, 635, 200, 665, 230], [0, 639, 300, 641, 330]], wh=(1080, 540))])
    plan = {p[0]: p[4] for p in tiles.sf_tile_plan(sp.images[0], 1080, 540)}
    assert set(plan) == {0, 440}
    assert len(plan[0]) == 1 and plan[0][0].startswith("0 ")
    cx, cy, w, h = map(float, plan[0][0].split()[1:])
    assert np.isclose(w * 640, 10, atol=0.01) and np.isclose(cx * 640, 635, atol=0.01) and np.isclose(h * 540, 30, atol=0.01)
    assert len(plan[440]) == 3                                              # all three lie fully inside the second tile


def test_build_sf_tiny(tmp_path, split_factory):
    import cv2
    img = tmp_path / "img000.jpg"
    cv2.imwrite(str(img), np.full((540, 1080, 3), 50, np.uint8))
    sp = split_factory([dict(boxes=[[0, 100, 100, 140, 130], [1, 700, 200, 740, 230]], ignore=[[0, 0, 20, 20]], wh=(1080, 540))])
    sp.images[0].path = img
    assert tiles.build_sf(sp, tmp_path / "out", workers=0) == (1, 2)
    assert len((tmp_path / "out/full/labels/img000.txt").read_text().splitlines()) == 2
    painted = cv2.imread(str(tmp_path / "out/tiles/images/img000_0_0.jpg"))
    assert painted.shape[:2] == (540, 640) and abs(int(painted[5, 5, 0]) - 114) <= 2


def test_build_d2_tiny(tmp_path, split_factory):
    import cv2
    img = tmp_path / "img000.jpg"
    cv2.imwrite(str(img), np.full((540, 1080, 3), 50, np.uint8))
    sp = split_factory([dict(boxes=[[0, 100, 100, 140, 130], [1, 630, 200, 660, 230]], ignore=[[0, 0, 20, 20]], wh=(1080, 540))])
    sp.images[0].path = img
    n_full, n_tiles = tiles.build_d2(sp, tmp_path / "out", workers=0)
    assert n_full == 1 and n_tiles >= 2
    lab = (tmp_path / "out/full/labels/img000.txt").read_text().splitlines()
    assert len(lab) == 2 and lab[0].startswith("0 ")
    painted = cv2.imread(str(tmp_path / "out/full/images/img000.jpg"))
    assert abs(int(painted[5, 5, 0]) - 114) <= 2                           # ignore region painted gray
    # box at x 630-660 is cut by the 640 tile [0, 640): 10/30 visible < 60% -> grayed, not labelled in that tile
    t0 = (tmp_path / "out/tiles/labels/img000_t640_0_0.txt").read_text().splitlines()
    assert len(t0) == 1

In [ ]:
%%writefile dronekit_src/tests/test_infer.py
import numpy as np
import pytest

from dronekit import infer


def test_box_cut_by_interior_slice_edge_is_removed():
    W, H = 1000, 1000
    sb = np.float32([0, 0, 600, 600])                    # slice whose right and bottom edges are interior
    d = np.float32([[550, 100, 600, 150, .9, 0],          # touches interior right edge -> cut object
                    [100, 560, 150, 599, .9, 0],          # within 2 px of interior bottom edge -> cut
                    [100, 100, 150, 150, .9, 0],          # well inside -> kept
                    [0, 300, 40, 340, .9, 0]])            # touches the image border (x=0), not interior -> kept
    out = infer.edge_filter(d, np.tile(sb, (len(d), 1)), W, H)
    assert out[:, 0].tolist() == [100, 0]


def test_edge_on_image_border_is_not_interior():
    W, H = 1000, 1000
    sb = np.float32([400, 400, 1000, 1000])               # right/bottom edges are the image border
    d = np.float32([[900, 900, 1000, 1000, .9, 0],        # touches border -> kept
                    [400, 500, 450, 550, .9, 0]])         # touches interior left edge -> removed
    out = infer.edge_filter(d, np.tile(sb, (len(d), 1)), W, H)
    assert out[:, :4].tolist() == [[900, 900, 1000, 1000]]


def test_full_pass_is_never_edge_filtered():
    d = np.float32([[0, 0, 10, 10, .9, 0]])
    assert infer.edge_filter(d, None, 100, 100) is d


def test_classwise_nms():
    d1 = np.float32([[0, 0, 100, 100, .9, 0], [0, 0, 100, 100, .8, 1]])
    d2 = np.float32([[2, 2, 100, 100, .7, 0]])            # duplicate of the class-0 box
    out = infer.merge([d1, d2], "NMS", "IOU", 0.6)
    assert sorted(out[:, 4].tolist()) == pytest.approx([0.8, 0.9])


def test_apply_setup_and_cache_roundtrip(tmp_path, split_factory):
    sp = split_factory([dict(wh=(1000, 1000)), dict(wh=(1000, 1000))])
    raw = {"full@1536": {"img000": (np.float32([[10, 10, 50, 50, .9, 0]]), None),
                         "img001": (np.zeros((0, 6), np.float32), None)},
           "s512@1280": {"img000": (np.float32([[11, 11, 50, 50, .8, 0], [480, 10, 512, 40, .9, 1]]),
                                    np.float32([[0, 0, 512, 512], [0, 0, 512, 512]])),
                         "img001": (np.float32([[100, 100, 130, 130, .5, 1]]), np.float32([[0, 0, 512, 512]]))}}
    dets, _ = infer.apply_setup(raw, sp, "best_sliced")
    # img000: slice dup of the full box removed by NMS; box at x2=512 cut by interior edge -> dropped
    assert dets["img000"].tolist() == [[10, 10, 50, 50, pytest.approx(.9), 0]]
    assert len(dets["img001"]) == 1
    whole, _ = infer.apply_setup(raw, sp, "whole1536")
    assert len(whole["img000"]) == 1

    # cache roundtrip through parquet
    import json
    import pandas as pd
    rows = []
    for p, per in raw.items():
        for k, (d, sb) in per.items():
            sb = np.full((len(d), 4), np.nan, np.float32) if sb is None else sb
            for r in np.c_[d, sb]:
                rows.append([k, p, *r])
    df = pd.DataFrame(rows, columns=infer.COLS)
    df.to_parquet(tmp_path / "c.parquet")
    (tmp_path / "c.json").write_text(json.dumps({"ms_per_img": {"full@1536": 30.0, "s512@1280": 400.0}}))
    raw2, meta = infer.load_cache(tmp_path / "c")
    d2, _ = infer.apply_setup(raw2, sp, "best_sliced")
    assert np.allclose(d2["img000"], dets["img000"])
    assert infer.setup_ms(meta, "best_sliced", 1.0) == pytest.approx(431.0)


def test_cache_with_zero_detections_keeps_its_passes(tmp_path, split_factory):
    import json
    import pandas as pd
    sp = split_factory([dict()])
    pd.DataFrame(columns=infer.COLS).astype({c: "float32" for c in infer.COLS[2:]}).to_parquet(tmp_path / "e.parquet")
    (tmp_path / "e.json").write_text(json.dumps({"passes": ["s512@1280", "full@1536"],
                                                 "ms_per_img": {"s512@1280": 1.0, "full@1536": 1.0}}))
    raw, _ = infer.load_cache(tmp_path / "e")
    dets, _ = infer.apply_setup(raw, sp, "best_sliced")
    assert len(dets["img000"]) == 0


def test_wbf_fuses_matching_boxes():
    a = np.float32([[100, 100, 200, 200, .9, 0]])
    b = np.float32([[102, 102, 202, 202, .7, 0]])
    out = infer.wbf([a, b], 1000, 1000, thr=0.65)
    assert len(out) == 1 and 100 <= out[0, 0] <= 102

In [ ]:
%%writefile dronekit_src/tests/test_score.py
import numpy as np
import pytest

from dronekit import data, score


def dets(rows):
    return np.array(rows, np.float32).reshape(-1, 6)


def test_box_inside_ignore_region_is_neither_tp_nor_fp(split_factory):
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200]], ignore=[[500, 500, 700, 700]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0],      # true positive
                         [550, 550, 600, 600, 0.95, 0]])}   # fully inside the ignore region, highest score
    m, pc, E = score.evaluate(sp, d)
    assert m["AP50"] == pytest.approx(1.0)                  # an FP ranked first would cut AP below 1
    pr = score.pr_at_conf(E, 0.0)
    assert (pr["tp"], pr["fp"], pr["n_gt"]) == (1, 0, 1)    # ignored det counted as neither
    cm = score.confusion_matrix(sp, d, conf=0.0)
    assert cm[-1].sum() == 0                                # no background (FP) entry either


def test_same_box_outside_ignore_region_is_fp(split_factory):
    # control: the identical detection without an ignore region does count as a false positive
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0], [550, 550, 600, 600, 0.95, 0]])}
    m, _, E = score.evaluate(sp, d)
    assert m["AP50"] < 1.0
    assert score.pr_at_conf(E, 0.0)["fp"] == 1


def test_without_ignore_turns_ignored_det_into_fp(split_factory):
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200]], ignore=[[500, 500, 700, 700]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0], [550, 550, 600, 600, 0.95, 0]])}
    assert score.evaluate(sp, d)[0]["AP50"] == pytest.approx(1.0)
    assert score.pr_at_conf(score.evaluate(data.without_ignore(sp), d)[2], 0.0)["fp"] == 1
    assert len(sp.images[0].ignore) == 1                    # the original split is untouched


def test_ignore_share_and_per_class_ap(split_factory):
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200]], ignore=[[500, 500, 700, 700]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0], [550, 550, 600, 600, 0.8, 0], [550, 550, 600, 600, 0.1, 0]])}
    assert score.ignore_share(sp, d, conf=0.25) == (0.5, 2)  # the 0.1 detection is below conf
    ap = score.per_class_ap(score.run_cocoeval(sp, d), sp.class_names)
    assert ap["a"] == pytest.approx(1.0) and np.isnan(ap["b"])


def test_by_sequence(split_factory):
    sp = split_factory([dict(), dict(), dict()])
    sp.images[2].sequence = "seq1"
    parts = data.by_sequence(sp)
    assert {k: len(v) for k, v in parts.items()} == {"seq0": 2, "seq1": 1}


def test_hand_computed_ap_three_boxes(split_factory):
    # 2 GT, 3 detections ranked: TP (0.9), FP (0.8), TP (0.7). Exact boxes -> same at every IoU.
    # precision = 1, 1/2, 2/3 at recall = 1/2, 1/2, 1. COCO envelope: 1 for r <= 0.5 (51 of the
    # 101 recall points), 2/3 for r in 0.51..1.00 (50 points).  AP = (51 + 50*2/3) / 101.
    sp = split_factory([dict(boxes=[[0, 100, 100, 200, 200], [0, 400, 400, 500, 500]])])
    d = {"img000": dets([[100, 100, 200, 200, 0.9, 0],
                         [700, 700, 800, 800, 0.8, 0],
                         [400, 400, 500, 500, 0.7, 0]])}
    m, pc, E = score.evaluate(sp, d)
    expected = (51 + 50 * 2 / 3) / 101
    assert pc["a"] == pytest.approx(expected, abs=1e-6)
    assert m["AP50"] == pytest.approx(expected, abs=1e-6)   # class "b" has no GT -> excluded
    assert m["AP"] == pytest.approx(expected, abs=1e-6)
    assert np.isnan(pc["b"])
    pr = score.pr_at_conf(E, 0.75)                          # keeps TP 0.9 and FP 0.8
    assert pr["precision"] == pytest.approx(0.5) and pr["recall"] == pytest.approx(0.5)


def test_size_buckets(split_factory):
    # one tiny (10x10), one small (20x20), one medium+ (40x40) object, all detected exactly
    sp = split_factory([dict(boxes=[[0, 0, 0, 10, 10], [0, 100, 100, 120, 120], [0, 300, 300, 340, 340]])])
    d = {"img000": dets([[0, 0, 10, 10, .9, 0], [100, 100, 120, 120, .8, 0], [300, 300, 340, 340, .7, 0]])}
    m, _, _ = score.evaluate(sp, d)
    for k in ("AP50_tiny", "AP50_small", "AP50_medium+", "AP50"):
        assert m[k] == pytest.approx(1.0), k


def test_max_dets_500_is_used(split_factory):
    # 200 GT objects, all detected: with maxDets 100 recall would cap at 0.5
    boxes = [[0, 20 * (i % 20), 20 * (i // 20) * 2, 20 * (i % 20) + 15, 20 * (i // 20) * 2 + 15] for i in range(200)]
    sp = split_factory([dict(boxes=boxes, wh=(1000, 1000))])
    d = {"img000": dets([[b[1], b[2], b[3], b[4], 0.5 + i / 1000, 0] for i, b in enumerate(boxes)])}
    m, _, E = score.evaluate(sp, d)
    assert m["AP50"] == pytest.approx(1.0)
    assert score.pr_at_conf(E, 0.0)["recall"] == pytest.approx(1.0)


def test_confusion_matrix_counts(split_factory):
    sp = split_factory([dict(boxes=[[0, 0, 0, 50, 50], [1, 100, 100, 150, 150], [1, 300, 300, 350, 350]])])
    d = {"img000": dets([[0, 0, 50, 50, .9, 0],            # a -> a
                         [100, 100, 150, 150, .8, 0],      # b predicted as a
                         [600, 600, 650, 650, .7, 1]])}    # background FP as b; GT at 300 missed
    cm = score.confusion_matrix(sp, d, conf=0.5)
    assert cm.tolist() == [[1, 0, 0], [1, 0, 1], [0, 1, 0]]
    assert np.allclose(score.normalize_rows(cm)[1], [0.5, 0, 0.5])

In [ ]:
%%writefile dronekit_src/tests/test_vlm.py
from dronekit import vlm


def test_parse_int_is_strict():
    assert vlm.parse_int("12") == 12 and vlm.parse_int(" 12.\n") == 12 and vlm.parse_int("0") == 0
    for bad in ("about 12", "12 cars", "There are 12.", "", None, "1,2", "-3"):
        assert vlm.parse_int(bad) is None


def test_parse_points_distinct_and_valid():
    assert vlm.parse_points("10,20\n30,40\n10,20\n") == 2
    assert vlm.parse_points("(10, 20)\n30,40\n30,") == 2           # last line cut by the token limit
    assert vlm.parse_points("NONE") == 0 and vlm.parse_points("none.") == 0
    assert vlm.parse_points("1500,20") is None and vlm.parse_points("I see cars") is None and vlm.parse_points("") is None


def test_prompts_name_the_target_and_boxes():
    assert "cars" in vlm.prompt_direct("car") and "integer" in vlm.prompt_direct("car")
    p = vlm.prompt_grounded("bus", [(1, 2, 3, 4, 0.5)])
    assert "1: 1,2,3,4 score 0.50" in p and "1 candidate" in p
    assert "(none)" in vlm.prompt_grounded("bus", [])

In [ ]:
# --- Unit tests (CPU, under a minute) ----------------------------------------
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "dronekit_src/tests"],
                   capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed: stop here and send the log"

In [ ]:
# --- Config ----------------------------------------------------------------
import os, sys, io, json, time, shutil, pickle, zipfile, logging, platform, subprocess
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, str(Path("dronekit_src").resolve()))
import dronekit
from dronekit import data, infer, score, vlm
logging.getLogger("sahi").setLevel(logging.WARNING)

ON_KAGGLE = Path("/kaggle").exists()
INPUT = Path(os.environ.get("DK_INPUT", "/kaggle/input"))
RAW = Path(os.environ.get("DK_UAVDT_RAW", "/tmp/UAVDT/raw"))
VIS = Path(os.environ.get("DK_DATA", "/tmp/VisDrone"))
CARPK_RAW = Path(os.environ.get("DK_CARPK_RAW", "/tmp/CARPK"))
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
OUT = WORK / "phase3_vlm"
CUDA = torch.cuda.is_available()
N_GPU = torch.cuda.device_count() if CUDA else 0
SMOKE = not CUDA

MODEL_ID = os.environ.get("DK_VLM_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct")
MAX_PIXELS = 1_500_000            # the processor resizes larger images to about this; the size used is logged per answer
TIME_H = 2.5                      # no new queries after this
MAX_TOKENS = dict(B=16, D=16, C=1024)
SEED = 0
UAV_VAL = ["M0201", "M0402", "M0604", "M0902", "M1006"]
VD_CONF = 0.41                    # YOLO26m-Tile best sliced, val best-F1 (experiments/phase0/results/basic_metrics.csv)
VD, UV = dronekit.VISDRONE_CLASSES, data.UAVDT_CLASSES
QUERIES_CSV = """dataset,split,image,sequence,target,count,count_bin,size_bin,density_bin,median_side,density,altitude
carpk,train,20160331_NTU_00002,20160331_NTU,car,4,1-8,large,dense,73.24530792236328,0.6830870509147644,
carpk,train,20160331_NTU_00003,20160331_NTU,car,4,1-8,large,dense,72.5262222290039,0.6828673481941223,
carpk,train,20160331_NTU_00004,20160331_NTU,car,4,1-8,large,dense,72.17435455322266,0.7019458413124084,
carpk,train,20160331_NTU_00005,20160331_NTU,car,3,1-8,large,dense,71.6658935546875,0.7259165048599243,
carpk,train,20160331_NTU_00008,20160331_NTU,car,12,9-20,large,dense,71.25630187988281,0.9426314830780029,
carpk,train,20160331_NTU_00033,20160331_NTU,car,11,9-20,large,dense,72.11102294921875,1.5727806091308594,
carpk,train,20160331_NTU_00044,20160331_NTU,car,29,21-50,large,dense,74.49832153320312,1.0664619207382202,
carpk,train,20160524_GF1_00041,20160524_GF1,car,37,21-50,large,dense,59.380130767822266,1.0073543787002563,
carpk,train,20160524_GF1_00087,20160524_GF1,car,37,21-50,large,dense,58.30952072143555,1.2206087112426758,
carpk,train,20160524_GF1_00111,20160524_GF1,car,39,21-50,large,dense,55.74943923950195,1.1473069190979004,
carpk,train,20160524_GF2_00004,20160524_GF2,car,26,21-50,large,dense,60.62042999267578,0.6886029839515686,
carpk,train,20160524_GF2_00007,20160524_GF2,car,27,21-50,large,dense,60.15812683105469,0.688454270362854,
carpk,train,20160524_GF2_00030,20160524_GF2,car,82,>50,large,dense,63.103004455566406,0.7313812375068665,
carpk,train,20160524_GF2_00090,20160524_GF2,car,60,>50,large,dense,53.28694152832031,0.8289494514465332,
carpk,train,20160524_GF2_00129,20160524_GF2,car,63,>50,large,dense,54.06477737426758,0.8109992146492004,
carpk,train,20160524_GF2_00162,20160524_GF2,car,54,>50,large,dense,53.05653381347656,0.8310476541519165,
carpk,train,20161029_NTU_00009,20161029_NTU,car,38,21-50,large,dense,78.91433715820312,0.9435349702835083,
carpk,train,20161029_NTU_00039,20161029_NTU,car,51,>50,large,dense,79.8248062133789,0.765663206577301,
carpk,train,20161029_NTU_00067,20161029_NTU,car,37,21-50,large,dense,77.38862609863281,0.8429127335548401,
carpk,train,20161029_NTU_00082,20161029_NTU,car,37,21-50,large,dense,81.06787109375,1.0275185108184814,
carpk,train,20161029_NTU_00124,20161029_NTU,car,24,21-50,large,dense,73.85796356201172,1.3986999988555908,
carpk,train,20161029_NTU_00129,20161029_NTU,car,19,9-20,large,dense,77.78174591064453,1.3971322774887085,
carpk,train,20161029_NTU_00132,20161029_NTU,car,18,9-20,large,dense,72.70831298828125,1.5333489179611206,
carpk,train,20161029_NTU_00144,20161029_NTU,car,11,9-20,large,dense,79.19596099853516,1.1404598951339722,
carpk,train,20161029_NTU_00161,20161029_NTU,car,52,>50,large,dense,74.64912414550781,0.7326322197914124,
carpk,train,20161030_GF1_00004,20161030_GF1,car,18,9-20,large,dense,80.84764099121094,1.2181044816970825,
carpk,train,20161030_GF1_00009,20161030_GF1,car,26,21-50,large,dense,60.31147384643555,1.387647032737732,
carpk,train,20161030_GF1_00078,20161030_GF1,car,51,>50,large,dense,74.45803833007812,0.7674033641815186,
carpk,train,20161030_GF1_00179,20161030_GF1,car,29,21-50,large,dense,65.24568939208984,1.0674550533294678,
carpk,train,20161030_GF2_00006,20161030_GF2,car,18,9-20,large,dense,85.09025573730469,0.741308867931366,
carpk,train,20161030_GF2_00008,20161030_GF2,car,18,9-20,large,dense,83.45048522949219,0.7608402967453003,
carpk,test,20161225_TPZ_00036,20161225_TPZ,car,99,>50,large,dense,51.70106506347656,0.8071838617324829,
carpk,test,20161225_TPZ_00064,20161225_TPZ,car,110,>50,large,dense,51.87001037597656,0.8121240139007568,
carpk,test,20161225_TPZ_00198,20161225_TPZ,car,4,1-8,large,dense,55.74933624267578,1.401879906654358,
carpk,test,20161225_TPZ_00203,20161225_TPZ,car,6,1-8,large,dense,54.05207824707031,0.762769341468811,
carpk,test,20161225_TPZ_00204,20161225_TPZ,car,7,1-8,large,dense,54.22176742553711,0.6950743794441223,
carpk,test,20161225_TPZ_00205,20161225_TPZ,car,8,1-8,large,dense,54.54261779785156,0.7348480820655823,
carpk,test,20161225_TPZ_00206,20161225_TPZ,car,9,9-20,large,dense,54.4058837890625,0.7062488198280334,
carpk,test,20161225_TPZ_00207,20161225_TPZ,car,9,9-20,large,dense,54.06477737426758,0.7208360433578491,
carpk,test,20161225_TPZ_00208,20161225_TPZ,car,11,9-20,large,dense,52.801513671875,0.7430101037025452,
carpk,test,20161225_TPZ_00290,20161225_TPZ,car,139,>50,large,dense,68.49817657470703,0.5823397636413574,
carpk,test,20161225_TPZ_00377,20161225_TPZ,car,95,>50,large,dense,51.61395263671875,0.7447975873947144,
carpk,test,20161225_TPZ_00388,20161225_TPZ,car,5,1-8,large,dense,52.99056625366211,1.1819020509719849,
carpk,test,20161225_TPZ_00389,20161225_TPZ,car,2,1-8,large,dense,54.81786346435547,0.8665539026260376,
carpk,test,20161225_TPZ_00391,20161225_TPZ,car,2,1-8,large,dense,42.18257141113281,0.6642044186592102,
carpk,test,20161225_TPZ_00394,20161225_TPZ,car,3,1-8,large,dense,54.60769271850586,0.6114910840988159,
carpk,test,20161225_TPZ_00397,20161225_TPZ,car,13,9-20,large,dense,54.442630767822266,0.706717848777771,
carpk,test,20161225_TPZ_00433,20161225_TPZ,car,127,>50,large,dense,51.2347526550293,0.725333571434021,
uavdt,test,M0203_img000251,M0203,car,7,1-8,large,dense,94.23374938964844,1.9109599590301514,low-alt
uavdt,test,M0203_img000361,M0203,car,8,1-8,large,dense,75.91250610351562,1.8182988166809082,low-alt
uavdt,test,M0203_img000571,M0203,car,6,1-8,large,mid,72.56353759765625,2.246363639831543,low-alt
uavdt,test,M0203_img000631,M0203,car,3,1-8,large,mid,104.76640319824219,2.6805579662323,low-alt
uavdt,test,M0203_img000671,M0203,car,2,1-8,large,mid,108.75421905517578,3.7756075859069824,low-alt
uavdt,test,M0205_img000201,M0205,car,7,1-8,large,mid,73.02054595947266,2.2393903732299805,low-alt
uavdt,test,M0205_img000401,M0205,car,10,9-20,large,mid,61.662193298339844,2.6193997859954834,low-alt
uavdt,test,M0205_img000531,M0205,car,6,1-8,large,mid,58.267250061035156,2.2543296813964844,low-alt
uavdt,test,M0205_img000561,M0205,car,5,1-8,large,mid,62.553977966308594,2.359193801879883,low-alt
uavdt,test,M0208_img000011,M0208,car,2,1-8,mid,mid,20.432270050048828,2.7197062969207764,medium-alt
uavdt,test,M0208_img000021,M0208,car,4,1-8,small,dense,15.968719482421875,1.915183186531067,medium-alt
uavdt,test,M0208_img000031,M0208,car,4,1-8,small,dense,16.073955535888672,1.9576382637023926,medium-alt
uavdt,test,M0208_img000041,M0208,car,4,1-8,small,mid,16.582439422607422,2.016862392425537,medium-alt
uavdt,test,M0208_img000051,M0208,car,3,1-8,small,dense,17.748239517211914,1.7419806718826294,medium-alt
uavdt,test,M0208_img000061,M0208,car,3,1-8,small,dense,18.76166343688965,1.6750587224960327,medium-alt
uavdt,test,M0208_img000071,M0208,car,5,1-8,small,dense,18.330303192138672,1.740257978439331,medium-alt
uavdt,test,M0208_img000091,M0208,car,4,1-8,small,mid,18.43110466003418,2.2905502319335938,medium-alt
uavdt,test,M0208_img000101,M0208,car,4,1-8,small,mid,19.286861419677734,2.2756850719451904,medium-alt
uavdt,test,M0208_img000121,M0208,car,3,1-8,mid,mid,22.203603744506836,2.6104040145874023,medium-alt
uavdt,test,M0208_img000141,M0208,car,2,1-8,mid,sparse,20.15209197998047,4.626163482666016,medium-alt
uavdt,test,M0208_img000231,M0208,car,3,1-8,mid,sparse,22.226110458374023,4.256199359893799,medium-alt
uavdt,test,M0208_img000241,M0208,car,2,1-8,mid,mid,22.274978637695312,3.6815342903137207,medium-alt
uavdt,test,M0209_img000131,M0209,car,11,9-20,large,mid,37.986839294433594,2.525958776473999,low-alt
uavdt,test,M0209_img000261,M0209,car,12,9-20,large,dense,35.727203369140625,1.6723157167434692,low-alt
uavdt,test,M0209_img000461,M0209,car,9,9-20,large,mid,36.98648452758789,2.625103235244751,low-alt
uavdt,test,M0209_img001231,M0209,car,14,9-20,large,dense,46.28133773803711,1.5468900203704834,low-alt
uavdt,test,M0403_img000021,M0403,car,62,>50,mid,dense,29.478683471679688,1.2025996446609497,medium-alt
uavdt,test,M0403_img000051,M0403,car,64,>50,mid,dense,29.563491821289062,1.2113691568374634,medium-alt
uavdt,test,M0403_img000061,M0403,car,64,>50,mid,dense,29.478683471679688,1.1557505130767822,medium-alt
uavdt,test,M0403_img000071,M0403,car,63,>50,mid,dense,29.664793014526367,1.1698482036590576,medium-alt
uavdt,test,M0403_img000091,M0403,car,63,>50,mid,dense,29.664793014526367,1.2201188802719116,medium-alt
uavdt,test,M0403_img000161,M0403,car,62,>50,mid,dense,29.444623947143555,1.2177726030349731,medium-alt
uavdt,test,M0403_img000211,M0403,car,63,>50,mid,dense,29.325756072998047,1.2037723064422607,medium-alt
uavdt,test,M0403_img000261,M0403,car,60,>50,mid,dense,29.563491821289062,1.2300710678100586,medium-alt
uavdt,test,M0403_img000271,M0403,car,60,>50,mid,dense,29.444623947143555,1.2193902730941772,medium-alt
uavdt,test,M0403_img000301,M0403,car,60,>50,mid,dense,29.359817504882812,1.1849031448364258,medium-alt
uavdt,test,M0403_img000401,M0403,car,58,>50,mid,dense,29.773948669433594,1.1409090757369995,medium-alt
uavdt,test,M0403_img000511,M0403,car,60,>50,mid,dense,29.7320613861084,1.1341698169708252,medium-alt
uavdt,test,M0601_img000021,M0601,car,28,21-50,small,dense,16.215839385986328,1.7877726554870605,high-alt
uavdt,test,M0601_img000061,M0601,car,29,21-50,small,dense,16.0,1.7993884086608887,high-alt
uavdt,test,M0601_img000271,M0601,car,28,21-50,small,dense,15.921613693237305,1.5754536390304565,high-alt
uavdt,test,M0602_img000071,M0602,car,29,21-50,small,dense,12.489995956420898,1.3436192274093628,medium-alt
uavdt,test,M0602_img000091,M0602,car,27,21-50,small,dense,12.489995956420898,1.471511960029602,medium-alt
uavdt,test,M0602_img000121,M0602,car,26,21-50,small,dense,12.449834823608398,1.6979373693466187,medium-alt
uavdt,test,M0602_img000251,M0602,car,37,21-50,small,dense,12.489995956420898,1.6785633563995361,medium-alt
uavdt,test,M0606_img000491,M0606,car,10,9-20,mid,mid,19.660327911376953,2.6943466663360596,medium-alt
uavdt,test,M0606_img001261,M0606,car,11,9-20,small,mid,14.866068840026855,2.5999817848205566,medium-alt
uavdt,test,M0606_img001281,M0606,car,10,9-20,small,mid,14.694417953491211,2.3782894611358643,medium-alt
uavdt,test,M0606_img001321,M0606,car,9,9-20,small,mid,18.439088821411133,2.460045337677002,medium-alt
uavdt,test,M0606_img001331,M0606,car,8,1-8,small,mid,17.93734359741211,2.478712797164917,medium-alt
uavdt,test,M0606_img001351,M0606,car,6,1-8,small,sparse,18.966339111328125,4.805145740509033,medium-alt
uavdt,test,M0606_img001361,M0606,car,6,1-8,small,mid,18.43332290649414,3.503784656524658,medium-alt
uavdt,test,M0606_img001371,M0606,car,6,1-8,small,mid,19.078495025634766,3.675748825073242,medium-alt
uavdt,test,M0701_img000051,M0701,car,31,21-50,large,dense,34.98571014404297,1.1922028064727783,high-alt
uavdt,test,M0701_img000181,M0701,car,47,21-50,mid,dense,29.698484420776367,1.2606488466262817,high-alt
uavdt,test,M0801_img000031,M0801,car,10,9-20,large,mid,62.18349075317383,2.0820677280426025,low-alt
uavdt,test,M0802_img000341,M0802,car,36,21-50,mid,dense,27.624446868896484,1.5601836442947388,low-alt
uavdt,test,M0802_img000391,M0802,car,34,21-50,mid,dense,28.038618087768555,1.738376259803772,low-alt
uavdt,test,M0802_img000421,M0802,car,34,21-50,mid,dense,27.312606811523438,1.7999012470245361,low-alt
uavdt,test,M0802_img000641,M0802,car,32,21-50,mid,dense,29.247806549072266,1.727803349494934,low-alt
uavdt,test,M0802_img000721,M0802,car,25,21-50,mid,dense,29.79932975769043,1.8276963233947754,low-alt
uavdt,test,M0802_img000831,M0802,car,22,21-50,mid,dense,27.221786499023438,1.99297297000885,low-alt
uavdt,test,M0802_img001001,M0802,car,15,9-20,mid,mid,31.74901580810547,2.5542237758636475,low-alt
uavdt,test,M0802_img001011,M0802,car,16,9-20,mid,mid,28.693748474121094,2.958955764770508,low-alt
uavdt,test,M0802_img001041,M0802,car,15,9-20,mid,mid,33.0454216003418,2.909437656402588,low-alt
uavdt,test,M0802_img001081,M0802,car,14,9-20,mid,mid,32.2462043762207,2.725975275039673,low-alt
uavdt,test,M1007_img000001,M1007,car,5,1-8,large,mid,49.04079818725586,2.487190008163452,medium-alt
uavdt,test,M1007_img000141,M1007,car,3,1-8,mid,sparse,26.267850875854492,4.245699405670166,medium-alt
uavdt,test,M1007_img000251,M1007,car,11,9-20,mid,dense,30.29851531982422,1.9486784934997559,medium-alt
uavdt,test,M1007_img000291,M1007,car,8,1-8,mid,mid,27.616252899169922,2.261549711227417,medium-alt
uavdt,test,M1007_img000301,M1007,car,8,1-8,mid,mid,28.48585319519043,2.3927223682403564,medium-alt
uavdt,test,M1007_img000311,M1007,car,8,1-8,mid,mid,30.914714813232422,2.237603187561035,medium-alt
uavdt,test,M1007_img000341,M1007,car,7,1-8,mid,dense,29.580398559570312,1.8937461376190186,medium-alt
uavdt,test,M1007_img000371,M1007,car,7,1-8,large,mid,34.2928581237793,2.0411484241485596,medium-alt
uavdt,test,M1007_img000441,M1007,car,7,1-8,large,mid,37.30951690673828,2.1934242248535156,medium-alt
uavdt,test,M1007_img000491,M1007,car,8,1-8,mid,mid,27.99854278564453,2.1373324394226074,medium-alt
uavdt,test,M1009_img000011,M1009,car,15,9-20,mid,dense,27.221315383911133,1.637227177619934,medium-alt
uavdt,test,M1009_img000151,M1009,car,22,21-50,mid,mid,27.221315383911133,2.2790112495422363,medium-alt
uavdt,test,M1009_img000511,M1009,car,8,1-8,mid,mid,27.12916374206543,2.9211666584014893,medium-alt
uavdt,test,M1101_img000051,M1101,car,12,9-20,large,mid,35.998077392578125,2.688594102859497,low-alt
uavdt,test,M1101_img000291,M1101,car,12,9-20,large,mid,33.93891143798828,2.0141196250915527,low-alt
uavdt,test,M1101_img000341,M1101,car,14,9-20,large,dense,37.44995880126953,1.9748706817626953,low-alt
uavdt,test,M1101_img000411,M1101,car,14,9-20,large,mid,34.436431884765625,2.24139404296875,low-alt
uavdt,test,M1101_img000501,M1101,car,9,9-20,large,mid,40.804412841796875,2.951469898223877,low-alt
uavdt,test,M1101_img000801,M1101,car,7,1-8,large,dense,64.8999252319336,1.7192482948303223,low-alt
uavdt,test,M1301_img000071,M1301,car,45,21-50,small,mid,11.832159996032715,2.2336337566375732,medium-alt
uavdt,test,M1301_img000441,M1301,car,38,21-50,small,mid,15.654753684997559,2.5295393466949463,medium-alt
uavdt,test,M1301_img000501,M1301,car,37,21-50,small,mid,16.492422103881836,2.436361074447632,medium-alt
uavdt,test,M1301_img000871,M1301,car,23,21-50,mid,mid,29.580398559570312,2.140028476715088,medium-alt
uavdt,test,M1301_img001021,M1301,car,20,9-20,mid,dense,24.919872283935547,1.8866865634918213,medium-alt
uavdt,test,M1301_img001141,M1301,car,22,21-50,mid,mid,19.899747848510742,2.0356273651123047,medium-alt
uavdt,test,M1302_img000161,M1302,car,26,21-50,mid,dense,22.405311584472656,1.296178936958313,medium-alt
uavdt,test,M1302_img000281,M1302,car,30,21-50,mid,dense,20.099504470825195,1.2773960828781128,medium-alt
uavdt,test,M1302_img000341,M1302,car,29,21-50,small,dense,15.874507904052734,1.5670955181121826,medium-alt
uavdt,test,M1302_img000491,M1302,car,35,21-50,small,dense,16.431676864624023,1.3895378112792969,medium-alt
uavdt,test,M1302_img000691,M1302,car,19,9-20,small,dense,16.492422103881836,1.597307801246643,medium-alt
uavdt,test,M1302_img000711,M1302,car,19,9-20,small,dense,16.492422103881836,1.6589179039001465,medium-alt
uavdt,test,M1303_img000021,M1303,car,30,21-50,large,dense,38.87014389038086,0.9881717562675476,low-alt
uavdt,test,M1303_img000061,M1303,car,30,21-50,large,dense,37.97613525390625,1.2281103134155273,low-alt
uavdt,test,M1303_img000091,M1303,car,30,21-50,large,dense,36.91167068481445,1.2448066473007202,low-alt
uavdt,test,M1303_img000101,M1303,car,29,21-50,large,dense,36.055511474609375,1.3986834287643433,low-alt
uavdt,test,M1303_img000161,M1303,car,31,21-50,large,dense,36.98648452758789,1.2375726699829102,low-alt
uavdt,test,M1303_img000171,M1303,car,28,21-50,large,dense,37.10040283203125,1.1806052923202515,low-alt
uavdt,test,M1303_img000181,M1303,car,27,21-50,large,dense,37.121421813964844,1.2992743253707886,low-alt
uavdt,test,M1303_img000241,M1303,car,26,21-50,large,dense,41.74205780029297,1.3056674003601074,low-alt
uavdt,test,M1303_img000251,M1303,car,27,21-50,large,dense,42.05948257446289,1.3014960289001465,low-alt
uavdt,test,M1303_img000271,M1303,car,24,21-50,large,dense,44.36743927001953,1.374941349029541,low-alt
uavdt,test,M1303_img000281,M1303,car,23,21-50,large,dense,43.47412872314453,1.281515121459961,low-alt
uavdt,test,M1303_img000331,M1303,car,19,9-20,large,dense,47.8643913269043,1.4741894006729126,low-alt
uavdt,test,M1401_img000041,M1401,car,14,9-20,mid,dense,24.586410522460938,1.4025237560272217,low-alt
uavdt,test,M1401_img000061,M1401,car,16,9-20,mid,dense,27.92331314086914,1.7399499416351318,low-alt
uavdt,test,M1401_img000071,M1401,car,16,9-20,mid,mid,28.64142417907715,2.0615930557250977,low-alt
uavdt,test,M1401_img000671,M1401,car,18,9-20,small,mid,19.233051300048828,2.447082281112671,low-alt
uavdt,test,M1401_img000731,M1401,car,16,9-20,mid,dense,22.890880584716797,1.8941564559936523,low-alt
uavdt,test,M1401_img000861,M1401,car,20,9-20,small,mid,18.73345375061035,2.744710922241211,low-alt
uavdt,test,M1401_img000871,M1401,car,19,9-20,small,mid,18.493242263793945,2.1150460243225098,low-alt
uavdt,test,M1401_img000891,M1401,car,19,9-20,small,mid,18.973665237426758,2.0623066425323486,low-alt
uavdt,test,M1401_img000931,M1401,car,17,9-20,small,dense,18.493242263793945,1.6779967546463013,low-alt
uavdt,test,M1401_img000941,M1401,car,18,9-20,small,mid,17.99304962158203,2.4118573665618896,low-alt
uavdt,test,M1401_img000961,M1401,car,18,9-20,small,mid,18.746952056884766,2.268510103225708,low-alt
visdrone,test-dev,0000006_02138_d_0000006,0000006,truck,26,21-50,large,dense,72.37254333496094,0.6593372821807861,
visdrone,test-dev,0000006_02138_d_0000006,0000006,car,3,1-8,large,dense,116.17228698730469,0.794722318649292,
visdrone,test-dev,0000011_01955_d_0000004,0000011,pedestrian,34,21-50,mid,dense,27.058074951171875,1.416772484779358,
visdrone,test-dev,0000011_01955_d_0000004,0000011,car,0,0,none,none,,,
visdrone,test-dev,0000063_02235_d_0000003,0000063,pedestrian,85,>50,small,dense,17.32050895690918,1.0389769077301025,
visdrone,test-dev,0000063_02235_d_0000003,0000063,car,0,0,none,none,,,
visdrone,test-dev,0000063_12500_d_0000014,0000063,pedestrian,127,>50,small,dense,15.29705810546875,1.0515550374984741,
visdrone,test-dev,0000063_12500_d_0000014,0000063,car,4,1-8,small,sparse,16.95181655883789,5.657470226287842,
visdrone,test-dev,0000073_01275_d_0000002,0000073,pedestrian,114,>50,mid,dense,27.38475799560547,0.8297085762023926,
visdrone,test-dev,0000073_01275_d_0000002,0000073,car,0,0,none,none,,,
visdrone,test-dev,0000074_02723_d_0000005,0000074,pedestrian,66,>50,large,dense,45.158504486083984,0.9364969730377197,
visdrone,test-dev,0000074_02723_d_0000005,0000074,car,0,0,none,none,,,
visdrone,test-dev,0000074_03738_d_0000007,0000074,pedestrian,75,>50,mid,dense,29.49576187133789,1.0056819915771484,
visdrone,test-dev,0000074_03738_d_0000007,0000074,car,0,0,none,none,,,
visdrone,test-dev,0000074_07850_d_0000015,0000074,pedestrian,133,>50,mid,dense,27.12932014465332,0.7975669503211975,
visdrone,test-dev,0000074_07850_d_0000015,0000074,car,2,1-8,large,sparse,109.81845092773438,4.878730773925781,
visdrone,test-dev,0000074_09738_d_0000019,0000074,pedestrian,86,>50,large,dense,45.33454895019531,0.7911317348480225,
visdrone,test-dev,0000074_09738_d_0000019,0000074,car,3,1-8,large,sparse,66.07571411132812,6.177211761474609,
visdrone,test-dev,0000074_10218_d_0000020,0000074,pedestrian,76,>50,large,dense,37.30249786376953,0.9376993179321289,
visdrone,test-dev,0000074_10218_d_0000020,0000074,car,2,1-8,large,dense,81.41384887695312,1.3479074239730835,
visdrone,test-dev,0000074_11362_d_0000022,0000074,pedestrian,100,>50,mid,dense,31.835350036621094,0.8968681693077087,
visdrone,test-dev,0000074_11362_d_0000022,0000074,car,1,1-8,large,none,41.89271926879883,,
visdrone,test-dev,0000074_15298_d_0000030,0000074,pedestrian,58,>50,mid,dense,30.370203018188477,1.1256604194641113,
visdrone,test-dev,0000074_15298_d_0000030,0000074,car,0,0,none,none,,,
visdrone,test-dev,0000074_15793_d_0000031,0000074,pedestrian,58,>50,large,dense,36.34418487548828,1.004935622215271,
visdrone,test-dev,0000074_15793_d_0000031,0000074,car,1,1-8,large,none,44.362144470214844,,
visdrone,test-dev,0000087_01140_d_0000004,0000087,pedestrian,86,>50,small,dense,15.55634880065918,1.3215699195861816,
visdrone,test-dev,0000087_01140_d_0000004,0000087,car,0,0,none,none,,,
visdrone,test-dev,0000127_00265_d_0000120,0000127,car,88,>50,large,dense,39.30632019042969,1.307147741317749,
visdrone,test-dev,0000164_00078_d_0000159,0000164,car,12,9-20,mid,mid,32.6739501953125,3.0249063968658447,
visdrone,test-dev,0000265_01500_d_0000004,0000265,car,29,21-50,mid,dense,23.237899780273438,1.7147973775863647,
visdrone,test-dev,0000265_03500_d_0000008,0000265,car,60,>50,mid,dense,22.714096069335938,1.3541783094406128,
visdrone,test-dev,0000272_01500_d_0000004,0000272,car,78,>50,large,dense,36.42308807373047,1.0409917831420898,
visdrone,test-dev,0000272_02500_d_0000006,0000272,car,61,>50,large,dense,37.0,1.317069172859192,
visdrone,test-dev,0000278_02351_d_0000002,0000278,car,51,>50,large,dense,67.26068878173828,0.7825822830200195,
visdrone,test-dev,0000302_03866_d_0000117,0000302,car,54,>50,large,dense,34.682106018066406,1.5281697511672974,
visdrone,test-dev,0000314_01000_d_0000144,0000314,pedestrian,5,1-8,mid,dense,29.580398559570312,0.9623158574104309,
visdrone,test-dev,0000314_01000_d_0000144,0000314,car,3,1-8,large,sparse,95.95310974121094,4.205840587615967,
visdrone,test-dev,0000353_02500_d_0000193,0000353,car,23,21-50,large,mid,37.947330474853516,2.5896754264831543,
visdrone,test-dev,0000353_03500_d_0000195,0000353,car,31,21-50,small,mid,18.439088821411133,2.0783815383911133,
visdrone,test-dev,0000353_05000_d_0000198,0000353,car,23,21-50,mid,mid,32.24903106689453,2.317206621170044,
visdrone,test-dev,9999938_00000_d_0000008,9999938,car,9,9-20,mid,mid,32.03123474121094,3.6761956214904785,
visdrone,test-dev,9999938_00000_d_0000019,9999938,car,58,>50,mid,dense,23.72491455078125,1.4292558431625366,
visdrone,test-dev,9999938_00000_d_0000035,9999938,pedestrian,21,21-50,small,sparse,13.266499519348145,5.048361778259277,
visdrone,test-dev,9999938_00000_d_0000035,9999938,car,10,9-20,large,mid,39.47804260253906,3.6286070346832275,
visdrone,test-dev,9999938_00000_d_0000042,9999938,pedestrian,15,9-20,small,dense,11.958260536193848,1.740533709526062,
visdrone,test-dev,9999938_00000_d_0000042,9999938,car,5,1-8,large,sparse,54.662601470947266,4.889126300811768,
visdrone,test-dev,9999938_00000_d_0000057,9999938,car,15,9-20,mid,mid,29.84962272644043,3.0708537101745605,
visdrone,test-dev,9999938_00000_d_0000062,9999938,pedestrian,6,1-8,small,sparse,15.837663650512695,9.186423301696777,
visdrone,test-dev,9999938_00000_d_0000062,9999938,car,3,1-8,large,mid,37.5233268737793,3.441466808319092,
visdrone,test-dev,9999938_00000_d_0000070,9999938,car,7,1-8,mid,sparse,29.49576187133789,5.498269081115723,
visdrone,test-dev,9999938_00000_d_0000073,9999938,pedestrian,52,>50,small,dense,14.316756248474121,1.9706411361694336,
visdrone,test-dev,9999938_00000_d_0000073,9999938,car,5,1-8,mid,sparse,22.494443893432617,7.045715808868408,
visdrone,test-dev,9999938_00000_d_0000083,9999938,pedestrian,25,21-50,mid,mid,19.748416900634766,2.2932302951812744,
visdrone,test-dev,9999938_00000_d_0000083,9999938,car,3,1-8,large,sparse,79.9124526977539,5.022801876068115,
visdrone,test-dev,9999938_00000_d_0000086,9999938,pedestrian,67,>50,small,mid,10.392304420471191,2.5534474849700928,
visdrone,test-dev,9999938_00000_d_0000086,9999938,car,5,1-8,mid,sparse,25.690465927124023,8.524737358093262,
visdrone,test-dev,9999938_00000_d_0000098,9999938,pedestrian,59,>50,small,dense,7.745966911315918,1.9816126823425293,
visdrone,test-dev,9999938_00000_d_0000098,9999938,car,1,1-8,large,none,47.916595458984375,,
visdrone,test-dev,9999938_00000_d_0000140,9999938,pedestrian,12,9-20,small,mid,18.452239990234375,3.519810438156128,
visdrone,test-dev,9999938_00000_d_0000140,9999938,car,6,1-8,large,mid,67.65350341796875,3.7612056732177734,
visdrone,test-dev,9999938_00000_d_0000171,9999938,pedestrian,54,>50,small,mid,8.485280990600586,3.867514133453369,
visdrone,test-dev,9999938_00000_d_0000171,9999938,car,13,9-20,small,sparse,19.183326721191406,4.456171035766602,
visdrone,test-dev,9999938_00000_d_0000183,9999938,car,26,21-50,large,dense,38.86365509033203,1.9434951543807983,
visdrone,test-dev,9999938_00000_d_0000197,9999938,car,41,21-50,large,dense,42.70831298828125,1.6281626224517822,
visdrone,test-dev,9999938_00000_d_0000202,9999938,pedestrian,61,>50,small,mid,6.0,2.3046352863311768,
visdrone,test-dev,9999938_00000_d_0000202,9999938,car,5,1-8,mid,sparse,30.049959182739258,11.06756591796875,
visdrone,test-dev,9999938_00000_d_0000209,9999938,pedestrian,160,>50,small,mid,6.324555397033691,2.9326443672180176,
visdrone,test-dev,9999938_00000_d_0000209,9999938,car,26,21-50,mid,dense,30.407726287841797,1.527351975440979,
visdrone,test-dev,9999938_00000_d_0000238,9999938,car,17,9-20,mid,mid,26.324893951416016,3.4600086212158203,
visdrone,test-dev,9999938_00000_d_0000278,9999938,car,18,9-20,mid,mid,25.631410598754883,4.129861831665039,
visdrone,test-dev,9999938_00000_d_0000287,9999938,car,21,21-50,mid,mid,26.153392791748047,3.8730132579803467,
visdrone,test-dev,9999938_00000_d_0000301,9999938,pedestrian,25,21-50,small,mid,9.797959327697754,2.845449686050415,
visdrone,test-dev,9999938_00000_d_0000301,9999938,car,10,9-20,mid,mid,31.332717895507812,2.8537137508392334,
visdrone,test-dev,9999938_00000_d_0000309,9999938,car,5,1-8,large,sparse,54.33230972290039,5.676271915435791,
visdrone,test-dev,9999938_00000_d_0000345,9999938,car,17,9-20,mid,mid,22.045408248901367,3.8496806621551514,
visdrone,test-dev,9999938_00000_d_0000348,9999938,car,9,9-20,mid,mid,31.30495262145996,4.106817245483398,
visdrone,test-dev,9999938_00000_d_0000353,9999938,car,19,9-20,mid,mid,23.237899780273438,4.091728210449219,
visdrone,test-dev,9999938_00000_d_0000374,9999938,car,53,>50,mid,mid,24.657655715942383,2.2873244285583496,
visdrone,test-dev,9999938_00000_d_0000382,9999938,car,23,21-50,mid,mid,25.377155303955078,3.208562135696411,
visdrone,test-dev,9999938_00000_d_0000389,9999938,car,22,21-50,small,sparse,12.349072456359863,4.343352794647217,
visdrone,test-dev,9999938_00000_d_0000418,9999938,car,14,9-20,mid,mid,21.908903121948242,3.3747572898864746,
visdrone,test-dev,9999938_00000_d_0000440,9999938,car,4,1-8,large,sparse,38.972686767578125,4.723295211791992,
visdrone,test-dev,9999938_00000_d_0000447,9999938,car,24,21-50,mid,mid,24.363235473632812,2.6681933403015137,
visdrone,test-dev,9999938_00000_d_0000459,9999938,car,11,9-20,large,sparse,36.83748245239258,4.444042682647705,
visdrone,test-dev,9999938_00000_d_0000474,9999938,car,14,9-20,large,mid,34.95582580566406,2.602738380432129,
visdrone,test-dev,9999938_00000_d_0000481,9999938,car,14,9-20,large,mid,36.35805892944336,2.3955583572387695,
visdrone,test-dev,9999938_00000_d_0000493,9999938,car,33,21-50,mid,mid,27.92848014831543,2.3981857299804688,
visdrone,test-dev,9999938_00000_d_0000496,9999938,car,26,21-50,mid,mid,29.149425506591797,2.5909340381622314,
visdrone,test-dev,9999938_00000_d_0000501,9999938,car,9,9-20,large,mid,33.49626922607422,3.1059632301330566,
visdrone,test-dev,9999938_00000_d_0000513,9999938,car,6,1-8,large,mid,66.67831420898438,4.099967956542969,
visdrone,test-dev,9999938_00000_d_0000514,9999938,car,7,1-8,large,sparse,49.74937057495117,4.9439239501953125,
visdrone,test-dev,9999947_00000_d_0000019,9999947,motor,51,>50,small,mid,16.881942749023438,3.0703577995300293,
visdrone,test-dev,9999947_00000_d_0000019,9999947,car,43,21-50,large,dense,37.66961669921875,1.5333071947097778,
visdrone,test-dev,9999947_00000_d_0000027,9999947,pedestrian,41,21-50,small,mid,18.493242263793945,2.9972620010375977,
visdrone,test-dev,9999947_00000_d_0000027,9999947,car,0,0,none,none,,,
visdrone,test-dev,9999952_00000_d_0000001,9999952,motor,17,9-20,small,sparse,15.968719482421875,4.845520496368408,
visdrone,test-dev,9999952_00000_d_0000001,9999952,car,11,9-20,mid,mid,28.142494201660156,2.7017979621887207,
visdrone,test-dev,9999952_00000_d_0000033,9999952,car,12,9-20,small,sparse,12.18465805053711,13.119921684265137,
visdrone,test-dev,9999952_00000_d_0000034,9999952,car,22,21-50,small,mid,14.472188949584961,2.813340663909912,
visdrone,test-dev,9999952_00000_d_0000036,9999952,car,13,9-20,small,sparse,12.845232963562012,5.210350036621094,
visdrone,test-dev,9999952_00000_d_0000050,9999952,car,12,9-20,large,dense,53.093955993652344,1.970779538154602,
visdrone,test-dev,9999952_00000_d_0000053,9999952,car,6,1-8,mid,sparse,31.83745765686035,5.9912519454956055,
visdrone,test-dev,9999952_00000_d_0000059,9999952,car,24,21-50,small,sparse,10.477226257324219,5.20720911026001,
visdrone,test-dev,9999952_00000_d_0000064,9999952,car,13,9-20,mid,sparse,22.045408248901367,4.434939861297607,
visdrone,test-dev,9999952_00000_d_0000075,9999952,car,56,>50,large,dense,33.49626922607422,1.398154854774475,
visdrone,test-dev,9999952_00000_d_0000077,9999952,car,7,1-8,mid,sparse,32.15586853027344,5.62863302230835,
visdrone,test-dev,9999952_00000_d_0000089,9999952,car,20,9-20,small,sparse,17.452987670898438,6.9381513595581055,
visdrone,test-dev,9999952_00000_d_0000100,9999952,car,8,1-8,mid,sparse,22.49443244934082,5.019148349761963,
visdrone,test-dev,9999952_00000_d_0000103,9999952,pedestrian,26,21-50,small,mid,7.93500280380249,2.7842507362365723,
visdrone,test-dev,9999952_00000_d_0000103,9999952,car,15,9-20,small,sparse,12.409673690795898,5.747418403625488,
visdrone,test-dev,9999952_00000_d_0000112,9999952,car,38,21-50,mid,dense,23.151512145996094,1.8555177450180054,
visdrone,test-dev,9999952_00000_d_0000133,9999952,car,51,>50,small,mid,17.748239517211914,2.3881726264953613,
visdrone,test-dev,9999952_00000_d_0000142,9999952,car,8,1-8,large,mid,37.48717498779297,4.147824764251709,
visdrone,test-dev,9999952_00000_d_0000143,9999952,car,25,21-50,large,mid,38.53569793701172,2.1882007122039795,
visdrone,test-dev,9999952_00000_d_0000157,9999952,van,21,21-50,large,dense,76.22335815429688,1.2034687995910645,
visdrone,test-dev,9999952_00000_d_0000157,9999952,car,19,9-20,large,dense,67.9705810546875,1.2683894634246826,
visdrone,test-dev,9999952_00000_d_0000160,9999952,car,28,21-50,large,dense,49.13247299194336,1.5367659330368042,
visdrone,test-dev,9999952_00000_d_0000193,9999952,car,36,21-50,small,mid,8.15192699432373,4.161845684051514,
visdrone,test-dev,9999952_00000_d_0000214,9999952,car,31,21-50,small,mid,12.845232963562012,2.605191230773926,
visdrone,test-dev,9999952_00000_d_0000229,9999952,truck,14,9-20,large,dense,82.19010162353516,0.6439404487609863,
visdrone,test-dev,9999952_00000_d_0000229,9999952,car,8,1-8,large,mid,40.40525817871094,3.340646266937256,
visdrone,test-dev,9999952_00000_d_0000237,9999952,car,8,1-8,mid,sparse,25.797286987304688,5.781776428222656,
visdrone,test-dev,9999952_00000_d_0000248,9999952,car,6,1-8,large,sparse,77.8331298828125,4.451298713684082,
visdrone,test-dev,9999952_00000_d_0000268,9999952,car,5,1-8,mid,sparse,22.847318649291992,5.3594794273376465,
visdrone,test-dev,9999952_00000_d_0000281,9999952,car,19,9-20,small,mid,17.8885440826416,2.9493699073791504,
visdrone,test-dev,9999952_00000_d_0000293,9999952,car,39,21-50,small,mid,10.954451560974121,3.012342691421509,
visdrone,test-dev,9999952_00000_d_0000318,9999952,car,8,1-8,large,mid,39.503135681152344,3.0827643871307373,
visdrone,test-dev,9999952_00000_d_0000330,9999952,pedestrian,7,1-8,small,sparse,12.649110794067383,7.802120685577393,
visdrone,test-dev,9999952_00000_d_0000330,9999952,car,6,1-8,mid,sparse,31.473411560058594,5.361248970031738,
visdrone,test-dev,9999952_00000_d_0000334,9999952,pedestrian,6,1-8,small,sparse,13.582067489624023,11.466163635253906,
visdrone,test-dev,9999952_00000_d_0000334,9999952,car,5,1-8,large,mid,45.166358947753906,3.1606431007385254,
visdrone,test-dev,9999952_00000_d_0000335,9999952,car,7,1-8,mid,sparse,26.70206069946289,6.180649757385254,
visdrone,test-dev,9999952_00000_d_0000338,9999952,car,2,1-8,large,sparse,59.56065368652344,4.740388870239258,
visdrone,test-dev,9999963_00000_d_0000031,9999963,car,14,9-20,large,dense,87.00453186035156,1.7853015661239624,
visdrone,test-dev,9999963_00000_d_0000039,9999963,car,20,9-20,large,dense,93.8956069946289,0.9505594372749329,
visdrone,test-dev,9999963_00000_d_0000041,9999963,car,26,21-50,large,dense,59.7789306640625,1.774562120437622,
visdrone,test-dev,9999963_00000_d_0000044,9999963,car,9,9-20,large,dense,128.62347412109375,1.673909068107605,
visdrone,test-dev,9999963_00000_d_0000045,9999963,motor,8,1-8,large,mid,62.836334228515625,2.620389461517334,
visdrone,test-dev,9999963_00000_d_0000045,9999963,car,6,1-8,large,dense,130.72882080078125,1.6782995462417603,
visdrone,test-dev,9999963_00000_d_0000065,9999963,car,22,21-50,large,dense,33.99630355834961,1.6140186786651611,
visdrone,test-dev,9999973_00000_d_0000002,9999973,car,15,9-20,large,dense,57.628116607666016,1.5055454969406128,
visdrone,test-dev,9999973_00000_d_0000057,9999973,car,35,21-50,mid,mid,30.8544979095459,2.9593052864074707,
visdrone,test-dev,9999973_00000_d_0000063,9999973,car,29,21-50,large,dense,57.982757568359375,1.613999843597412,
visdrone,test-dev,9999973_00000_d_0000094,9999973,car,10,9-20,large,mid,56.894622802734375,2.3163087368011475,
visdrone,test-dev,9999973_00000_d_0000131,9999973,car,33,21-50,large,dense,38.66522979736328,1.8527858257293701,
visdrone,test-dev,9999973_00000_d_0000147,9999973,motor,19,9-20,small,mid,17.14642906188965,4.050860404968262,
visdrone,test-dev,9999973_00000_d_0000147,9999973,car,17,9-20,large,dense,59.766212463378906,1.991629719734192,
visdrone,test-dev,9999976_00000_d_0000003,9999976,pedestrian,5,1-8,small,sparse,9.165151596069336,14.636469841003418,
visdrone,test-dev,9999976_00000_d_0000003,9999976,car,2,1-8,small,sparse,18.11149024963379,19.284835815429688,
visdrone,test-dev,9999976_00000_d_0000005,9999976,car,15,9-20,mid,dense,25.41653060913086,1.8402262926101685,
visdrone,test-dev,9999976_00000_d_0000018,9999976,car,1,1-8,large,none,45.82575607299805,,
visdrone,test-dev,9999976_00000_d_0000021,9999976,pedestrian,9,9-20,small,mid,15.874507904052734,4.125337600708008,
visdrone,test-dev,9999976_00000_d_0000021,9999976,car,6,1-8,large,mid,38.164276123046875,2.2803831100463867,
visdrone,test-dev,9999976_00000_d_0000026,9999976,car,7,1-8,large,dense,57.00876998901367,1.6601308584213257,
visdrone,test-dev,9999976_00000_d_0000028,9999976,car,6,1-8,small,mid,8.214428901672363,3.2519094944000244,
visdrone,test-dev,9999976_00000_d_0000030,9999976,pedestrian,5,1-8,small,mid,16.248077392578125,2.5985610485076904,
visdrone,test-dev,9999976_00000_d_0000030,9999976,car,0,0,none,none,,,
visdrone,test-dev,9999976_00000_d_0000034,9999976,pedestrian,5,1-8,small,sparse,13.747727394104004,13.821805953979492,
visdrone,test-dev,9999976_00000_d_0000034,9999976,car,1,1-8,small,none,16.733200073242188,,
visdrone,test-dev,9999979_00000_d_0000016,9999979,motor,80,>50,mid,dense,21.071308135986328,1.0220420360565186,
visdrone,test-dev,9999979_00000_d_0000016,9999979,car,27,21-50,large,dense,50.5272216796875,1.3136790990829468,
visdrone,test-dev,9999979_00000_d_0000023,9999979,car,87,>50,large,dense,33.911651611328125,0.9552777409553528,
visdrone,test-dev,9999979_00000_d_0000026,9999979,car,84,>50,mid,dense,30.041637420654297,1.075521469116211,
visdrone,test-dev,9999979_00000_d_0000027,9999979,car,10,9-20,mid,mid,27.886375427246094,2.5871102809906006,
visdrone,test-dev,9999979_00000_d_0000044,9999979,car,63,>50,mid,dense,22.847318649291992,1.2322542667388916,
visdrone,test-dev,9999986_00000_d_0000023,9999986,car,4,1-8,large,mid,92.5086441040039,2.08748459815979,
visdrone,test-dev,9999986_00000_d_0000024,9999986,car,2,1-8,large,sparse,48.2955207824707,4.247287750244141,
visdrone,test-dev,9999986_00000_d_0000032,9999986,bicycle,4,1-8,mid,mid,29.392444610595703,2.906829595565796,
visdrone,test-dev,9999986_00000_d_0000032,9999986,car,2,1-8,large,sparse,96.04828643798828,4.74320650100708,
visdrone,test-dev,9999986_00000_d_0000034,9999986,pedestrian,7,1-8,small,dense,14.387495040893555,1.8973065614700317,
visdrone,test-dev,9999986_00000_d_0000034,9999986,car,1,1-8,large,none,80.48602294921875,,
visdrone,test-dev,9999986_00000_d_0000036,9999986,car,7,1-8,large,mid,68.84765625,2.9190497398376465,
visdrone,test-dev,9999986_00000_d_0000043,9999986,pedestrian,8,1-8,small,mid,17.029386520385742,2.0950064659118652,
visdrone,test-dev,9999986_00000_d_0000043,9999986,car,7,1-8,large,mid,59.74947738647461,2.3466103076934814,
visdrone,test-dev,9999986_00000_d_0000050,9999986,pedestrian,3,1-8,mid,sparse,19.595918655395508,4.773895263671875,
visdrone,test-dev,9999986_00000_d_0000050,9999986,car,0,0,none,none,,,
visdrone,test-dev,9999996_00000_d_0000010,9999996,car,20,9-20,small,dense,14.91619873046875,0.9577111601829529,
"""
Q = pd.read_csv(io.StringIO(QUERIES_CSV))
assert Q[["dataset", "image", "target"]].duplicated().sum() == 0
Q = Q.sample(frac=1, random_state=SEED).reset_index(drop=True)      # the query order of every run; qid is the position in it
Q["qid"] = np.arange(len(Q))
# Rerun of the queries that ran out of GPU memory in the first full run; [] runs every query. EXPECT checks the order is the same.
EXPECT = {0: '9999976_00000_d_0000034|car', 2: '9999973_00000_d_0000147|car', 8: '0000074_11362_d_0000022|car', 14: '9999938_00000_d_0000459|car', 18: '9999938_00000_d_0000073|car', 22: '0000353_02500_d_0000193|car', 23: '0000265_03500_d_0000008|car', 26: '9999986_00000_d_0000043|car', 30: '9999973_00000_d_0000063|car', 32: '0000074_15298_d_0000030|car', 38: '9999938_00000_d_0000278|car', 39: '0000272_01500_d_0000004|car', 42: '9999938_00000_d_0000301|pedestrian', 50: '0000074_03738_d_0000007|car', 52: '0000011_01955_d_0000004|car', 60: '9999938_00000_d_0000042|pedestrian', 62: '9999938_00000_d_0000062|car', 66: '9999938_00000_d_0000202|car', 79: '0000127_00265_d_0000120|car', 84: '20160524_GF2_00030|car', 86: '9999952_00000_d_0000281|car', 88: '20161029_NTU_00039|car', 90: '9999938_00000_d_0000062|pedestrian', 91: '0000074_15298_d_0000030|pedestrian', 92: '9999938_00000_d_0000035|car', 96: '9999952_00000_d_0000050|car', 104: '20160524_GF1_00087|car', 106: '9999938_00000_d_0000086|car', 108: '9999952_00000_d_0000036|car', 109: '0000073_01275_d_0000002|pedestrian', 110: '9999938_00000_d_0000493|car', 112: '9999938_00000_d_0000197|car', 116: '9999976_00000_d_0000030|car', 117: '9999938_00000_d_0000374|car', 122: '9999952_00000_d_0000001|motor', 128: '9999986_00000_d_0000034|pedestrian', 136: '9999979_00000_d_0000016|motor', 140: '9999938_00000_d_0000098|pedestrian', 142: '20161030_GF1_00078|car', 144: '9999996_00000_d_0000010|car', 148: '9999938_00000_d_0000070|car', 150: '9999938_00000_d_0000171|pedestrian', 152: '0000353_05000_d_0000198|car', 153: '9999952_00000_d_0000293|car', 154: '9999986_00000_d_0000032|car', 155: '9999938_00000_d_0000019|car', 162: '9999952_00000_d_0000112|car', 164: '9999938_00000_d_0000209|pedestrian', 166: '9999952_00000_d_0000338|car', 168: '0000314_01000_d_0000144|car', 170: '9999979_00000_d_0000027|car', 176: '9999986_00000_d_0000050|pedestrian', 178: '9999938_00000_d_0000042|car', 184: '20161029_NTU_00009|car', 188: '9999938_00000_d_0000440|car', 196: '9999952_00000_d_0000077|car', 197: '0000087_01140_d_0000004|pedestrian', 200: '9999973_00000_d_0000057|car', 201: '9999938_00000_d_0000073|pedestrian', 202: '9999976_00000_d_0000026|car', 203: '20161225_TPZ_00290|car', 205: '20160524_GF2_00090|car', 211: '20161029_NTU_00161|car', 214: '0000074_11362_d_0000022|pedestrian', 215: '0000074_09738_d_0000019|pedestrian', 216: '9999976_00000_d_0000021|pedestrian', 218: '9999986_00000_d_0000024|car', 220: '9999952_00000_d_0000318|car', 222: '0000074_07850_d_0000015|pedestrian', 235: '20161225_TPZ_00377|car', 236: '20160524_GF2_00129|car', 240: '9999952_00000_d_0000064|car', 242: '9999976_00000_d_0000034|pedestrian', 243: '9999979_00000_d_0000023|car', 244: '9999986_00000_d_0000034|car', 246: '9999938_00000_d_0000083|car', 249: '9999947_00000_d_0000019|motor', 250: '9999952_00000_d_0000142|car', 252: '0000074_10218_d_0000020|car', 254: '9999952_00000_d_0000100|car', 255: '0000074_10218_d_0000020|pedestrian', 258: '0000272_02500_d_0000006|car', 261: '0000074_02723_d_0000005|pedestrian', 262: '9999938_00000_d_0000496|car', 264: '9999976_00000_d_0000005|car', 266: '9999963_00000_d_0000031|car', 270: '0000353_03500_d_0000195|car', 274: '0000074_03738_d_0000007|pedestrian', 276: '9999963_00000_d_0000065|car', 278: '9999947_00000_d_0000027|pedestrian', 279: '0000302_03866_d_0000117|car', 282: '9999952_00000_d_0000330|pedestrian', 286: '9999952_00000_d_0000034|car', 290: '9999963_00000_d_0000045|motor', 293: '9999952_00000_d_0000075|car', 294: '9999963_00000_d_0000041|car', 304: '9999938_00000_d_0000348|car', 305: '0000278_02351_d_0000002|car', 306: '20161225_TPZ_00036|car', 307: '20161225_TPZ_00064|car', 310: '9999976_00000_d_0000021|car', 312: '9999973_00000_d_0000131|car', 314: '9999952_00000_d_0000160|car', 316: '0000073_01275_d_0000002|car', 325: '9999979_00000_d_0000026|car', 326: '0000063_12500_d_0000014|pedestrian', 327: '9999979_00000_d_0000044|car', 330: '9999973_00000_d_0000147|motor', 334: '9999952_00000_d_0000268|car', 336: '9999952_00000_d_0000103|pedestrian', 338: '20160524_GF1_00111|car', 340: '9999938_00000_d_0000447|car', 342: '0000074_15793_d_0000031|pedestrian', 344: '20161225_TPZ_00433|car', 345: '0000063_02235_d_0000003|pedestrian'}
ONLY_QIDS = sorted(EXPECT)
FALLBACK_PIXELS = [1_000_000, 700_000]   # only after an out-of-memory error at MAX_PIXELS, then at the same size again
if ONLY_QIDS:
    got = {int(q.qid): f"{q.image}|{q.target}" for q in Q.itertuples() if q.qid in EXPECT}
    assert got == EXPECT, "query order differs from the first run: stop and send this"
    Q = Q[Q.qid.isin(ONLY_QIDS)].reset_index(drop=True)
for d in ("results", "figures", "logs"):
    (OUT / d).mkdir(parents=True, exist_ok=True)
print(len(Q), "queries", Q.dataset.value_counts().to_dict(), "| GPUs", N_GPU, "| SMOKE", SMOKE)

In [ ]:
# --- Download + extract the official release (skipped if an attached copy or a previous extract exists) ---
import subprocess, tarfile, zipfile
# Team copies of the official UAVDT-M Google Drive files (originals: https://sites.google.com/view/daweidu/projects/uavdt;
# the original IDs hit Drive's download quota: images 1m8KA6oPIRK_Iwt9TYFquC87vBc_8wRVc, toolkit 19498uJd7T9w4quwnQEy62nibt3uyT9pq, attrs 1qjipvuk3XE3qU3udluQRRcYuiKzhMXB1)
DRIVE = {"images": "1F9BN_AE5kqo157O2RaH91g4filLIYpRr",       # UAV-benchmark-M (frames); team copy of the official file
         "toolkit": "15N2S1QGlLxwYXHg2RtFyPw_b1q5rxK6t",      # UAV-benchmark-MOTD_v1.0 (GT incl. *_gt_whole / *_gt_ignore)
         "attrs": "1vMYo-s7CZ0QxQsY9DeIBlYjWgA2cv-Q9"}        # M_attr (sequence attributes; train/test folders = split)

def extract(archive, dest):
    dest.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        zipfile.ZipFile(archive).extractall(dest)
    elif tarfile.is_tarfile(archive):
        tarfile.open(archive).extractall(dest)
    else:                                            # e.g. .rar / .7z
        for cmd in (["7z", "x", "-y", f"-o{dest}", str(archive)], ["unrar", "x", "-o+", str(archive), str(dest)]):
            if shutil.which(cmd[0]) and subprocess.run(cmd, capture_output=True).returncode == 0:
                return
        raise RuntimeError(f"cannot extract {archive}: unknown format; send this message back")

def seq_dir_root(root, seq="M0101"):
    hits = [p for p in Path(root).rglob(seq) if p.is_dir() and any(p.glob("img*.jpg"))] if Path(root).exists() else []
    return hits[0].parent if hits else None

def drive_download(key, dest):
    # The direct link first: gdown gives up on Drive's "too many users" page even when this link still serves the file
    url = f"https://drive.usercontent.google.com/download?id={DRIVE[key]}&export=download&confirm=t"
    subprocess.run(["curl", "-sS", "-L", "--fail", "--retry", "3", "-C", "-", "-o", str(dest), url])
    if dest.exists() and dest.stat().st_size > 0:
        with open(dest, "rb") as fh:
            if fh.read(1) != b"<":          # "<" = Drive's error page, not an archive
                return dest
    import gdown
    dest.unlink(missing_ok=True)
    try:
        return Path(gdown.download(id=DRIVE[key], output=str(dest), quiet=True))
    except Exception as e:
        raise RuntimeError(f"Google Drive refused {key}: {str(e)[:200]}. Upload it as a private Kaggle Dataset and attach it, or send this message back.")

def fetch(key, name):
    d = RAW / name
    if d.exists() and any(d.iterdir()):
        return d
    (RAW / "zips").mkdir(parents=True, exist_ok=True)
    print("downloading", name, "from Google Drive ...")
    f = drive_download(key, RAW / "zips" / f"{name}.bin")
    print("  got", f"{f.stat().st_size / 1e9:.2f} GB; extracting")
    extract(f, d); f.unlink()
    return d

def locate(pattern, key, name):
    # an attached Kaggle copy first, then a previous extract in /tmp, else download
    for root in (INPUT, RAW):
        hit = next(iter(sorted(root.rglob(pattern))), None) if root.exists() else None
        if hit:
            return hit
    return next(fetch(key, name).rglob(pattern))

IMG_ROOT = seq_dir_root(INPUT) or seq_dir_root(RAW / "images") or seq_dir_root(fetch("images", "images"))
GT_DIR = locate("M0101_gt_whole.txt", "toolkit", "toolkit").parent
ATTR = locate("M0101_attr.txt", "attrs", "attrs").parent.parent        # M_attr/ (holds train/ and test/)
print("images:", IMG_ROOT, "\nGT:", GT_DIR, "\nattributes:", ATTR)

In [ ]:
# --- CARPK: an attached copy, a previous extract, or the authors' Drive file with the password from Kaggle Secrets ---
CARPK_ID = "0BwSzgS8Mm48Ud2h2dW40Wko3a1E"; CARPK_KEY = "0-34K_uP-vYM7EWq0Q2iIVaw"
CARPK_IMAGES = 1448                  # 989 train + 459 test, read from ImageSets in the real archive

def carpk_password():
    if os.environ.get("DK_CARPK_PW"):
        return os.environ["DK_CARPK_PW"]
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret("CARPK_PASSWORD")

def find_carpk():
    for root in (INPUT, CARPK_RAW):
        hit = next(iter(sorted(p for p in root.rglob("CARPK_devkit") if p.is_dir())), None) if root.exists() else None
        if hit:
            return hit
    CARPK_RAW.mkdir(parents=True, exist_ok=True)
    z = CARPK_RAW / "datasets.zip"
    url = f"https://drive.usercontent.google.com/download?id={CARPK_ID}&resourcekey={CARPK_KEY}&export=download&confirm=t"
    subprocess.run(["curl", "-sS", "-L", "-C", "-", "-o", str(z), url], check=True)
    print(f"CARPK archive: {z.stat().st_size / 1e9:.2f} GB")
    parts = [f"datasets/CARPK_devkit/data/{s}/*" for s in ("Images", "Annotations", "ImageSets")]     # the parent pattern matched nothing in a local test
    r = subprocess.run(["unzip", "-q", "-o", "-P", carpk_password(), str(z), *parts, "-d", str(CARPK_RAW)], capture_output=True, text=True)
    root = CARPK_RAW / "datasets/CARPK_devkit"
    assert r.returncode == 0 and len(list(root.rglob("*.png"))) == CARPK_IMAGES, "CARPK did not extract: check the CARPK_PASSWORD secret (" + r.stderr[-300:].replace("\n", " ") + ")"
    z.unlink()
    return root

CARPK_ROOT = find_carpk()
print("CARPK:", CARPK_ROOT)

In [ ]:
# --- The query images, and a detector run for system D -------------------------------------------------------------
from ultralytics import YOLO
from ultralytics.utils import LOGGER
LOGGER.setLevel(logging.ERROR)
ATTRS, OFFICIAL = data.read_uavdt_attrs(ATTR)
vis = data.load_visdrone(VIS / "VisDrone2019-DET-test-dev" if (VIS / "VisDrone2019-DET-test-dev/annotations").exists() else data.download_visdrone(VIS, "test-dev"), "test-dev")
uv_seqs = sorted(s for s, v in OFFICIAL.items() if v == "test")
uv = data.load_uavdt(IMG_ROOT, GT_DIR, ATTR, uv_seqs, 10, "test")
cp = data.load_carpk(CARPK_ROOT, "test").images + data.load_carpk(CARPK_ROOT, "train").images
REC = {("visdrone", r.key): r for r in vis.images} | {("uavdt", r.key): r for r in uv.images} | {("carpk", r.key): r for r in cp}
if SMOKE:
    Q = Q[[k in REC for k in zip(Q.dataset, Q.image)]].reset_index(drop=True) if any(k in REC for k in zip(Q.dataset, Q.image)) else Q.iloc[:0]
    if not len(Q):
        keys = list(REC)[:4]
        Q = pd.DataFrame([dict(dataset=d, split="x", image=i, sequence=REC[(d, i)].sequence, target="car", count=len(REC[(d, i)].boxes), count_bin="x", size_bin="x", density_bin="x", qid=n) for n, (d, i) in enumerate(keys)])
missing = [k for k in zip(Q.dataset, Q.image) if k not in REC]
assert not missing, f"{len(missing)} query images are missing, e.g. {missing[:3]}: stop and send the log"

def weights(name):
    hits = sorted(INPUT.rglob(name))
    assert hits, f"attach {name} (the weights Dataset from phase1_eval)"
    return str(hits[0])

device = dict(cuda=CUDA, half=CUDA, device=0 if CUDA else "cpu")
def run(model_file, recs, passes, setup):
    sp = data.Split("x", "x", [], recs)
    tmp = OUT / "logs" / f"d_{Path(model_file).stem}_{len(recs)}"
    infer.run_and_cache(YOLO(model_file), sp, passes, tmp, **device)
    return infer.apply_setup(infer.load_cache(tmp)[0], sp, setup)[0]

W_VD, W_UV = weights("yolo26m_tile_visdrone.pt"), weights("yolo26m_tile_uavdt.pt")
keys = sorted(set(zip(Q.dataset, Q.image)))
# UAVDT threshold: best F1 on UAVDT val at the selection pass; VisDrone threshold from Phase 0
uv_val = data.load_uavdt(IMG_ROOT, GT_DIR, ATTR, UAV_VAL if ON_KAGGLE else [s for s, v in OFFICIAL.items() if v == "train"], 10, "val")
WHOLE = dict(passes=["full@1024"], method=None)
d_val = run(W_UV, uv_val.images if ON_KAGGLE else uv_val.images[:3], ["full@1024"], WHOLE)
_, _, E = score.evaluate(uv_val if ON_KAGGLE else uv_val.subset(3), d_val)
UV_CONF = score.best_f1_conf(E) if E is not None else 0.25
print("thresholds: VisDrone model", VD_CONF, "| UAVDT model", UV_CONF)
DET = {}
vd_recs = [REC[k] for k in keys if k[0] != "uavdt"]
uv_recs = [REC[k] for k in keys if k[0] == "uavdt"]
BEST = infer.SETUPS["best_sliced"]
if vd_recs:
    DET.update(run(W_VD, vd_recs, BEST["passes"], BEST))
if uv_recs:
    DET.update(run(W_UV, uv_recs, ["full@1024"], WHOLE))

def boxes_for(dataset, image, target):
    # detector candidates for one query, highest score first. CARPK has no model of its own: the VisDrone model, with van as car.
    d = DET[image]
    names, conf = (UV, UV_CONF) if dataset == "uavdt" else (VD, VD_CONF)
    ids = [names.index(target)] + ([names.index("van")] if dataset == "carpk" and target == "car" else [])
    d = d[np.isin(d[:, 5].astype(int), ids) & (d[:, 4] >= conf)]
    return d[np.argsort(-d[:, 4])][:, :5].tolist()

In [ ]:
# --- Worker: one model per GPU, append after every answer, skip what is already written ---------------------------
import gc
gc.collect()
if CUDA:
    torch.cuda.empty_cache()                 # the detector above ran on GPU 0; give that memory back before the VLM workers start
    print("GPU 0 memory still reserved by this kernel:", round(torch.cuda.memory_reserved(0) / 2**30, 2), "GB")
JOB = []
for q in Q.itertuples():
    r = REC[(q.dataset, q.image)]
    JOB.append(dict(qid=int(q.qid), dataset=q.dataset, image=q.image, sequence=q.sequence, target=q.target, count=int(q.count), path=str(r.path), boxes=boxes_for(q.dataset, q.image, q.target), width=r.width, height=r.height))
pickle.dump(JOB, open(OUT / "logs" / "jobs.pkl", "wb"))
pd.DataFrame([dict(qid=j["qid"], dataset=j["dataset"], image=j["image"], target=j["target"], count=j["count"], detector_count=len(j["boxes"])) for j in JOB]).to_csv(OUT / "results" / "detector_counts_for_d.csv", index=False)

WORKER = r"""
import gc, json, os, pickle, re, sys, time
from pathlib import Path
gpu, n_gpu, out, model_id, max_pixels, deadline, limit = sys.argv[1], int(sys.argv[2]), Path(sys.argv[3]), sys.argv[4], int(sys.argv[5]), float(sys.argv[6]), int(sys.argv[7])
fallback = [int(v) for v in sys.argv[8].split(",") if v]
os.environ["CUDA_VISIBLE_DEVICES"] = "" if gpu == "cpu" else gpu
sys.path.insert(0, "dronekit_src")
import cv2, torch
from PIL import Image
from dronekit import vlm
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
MAX_TOKENS = dict(B=16, D=16, C=1024)
kw = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16), device_map={"": 0}) if gpu != "cpu" else {}
model = AutoModelForImageTextToText.from_pretrained(model_id, **kw).eval()
proc = AutoProcessor.from_pretrained(model_id, max_pixels=max_pixels)

def ask(path, prompt, n, px):
    msgs = [dict(role="user", content=[dict(type="image", image=str(path), max_pixels=px), dict(type="text", text=prompt)])]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(msgs)
    x = proc(text=[text], images=images, videos=videos, padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        y = model.generate(**x, max_new_tokens=n, do_sample=False)
    new = y[0, x.input_ids.shape[1]:]
    grid = x.image_grid_thw[0].tolist() if "image_grid_thw" in x else [0, 0, 0]
    return proc.decode(new, skip_special_tokens=True), int(len(new)), [int(grid[2]) * 14, int(grid[1]) * 14]

def grounded_image(j):
    im = cv2.imread(j["path"])
    for i, (x1, y1, x2, y2, s) in enumerate(j["boxes"]):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        if len(j["boxes"]) <= 60:
            cv2.putText(im, str(i + 1), (int(x1), max(int(y1) - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
    f = out / "logs" / f"g_{gpu}.jpg"
    cv2.imwrite(str(f), im)
    return f

fn = out / "results" / f"answers_{gpu}.jsonl"
done = set()
if fn.exists():
    done = {(r["qid"], r["system"]) for r in map(json.loads, fn.read_text().splitlines())}
jobs = pickle.load(open(out / "logs" / "jobs.pkl", "rb"))
jobs = [j for k, j in enumerate(jobs) if k % n_gpu == (0 if gpu == "cpu" else int(gpu))][:limit or None]      # by position, so a rerun subset is split evenly
with open(fn, "a") as f:
    for j in jobs:
        if time.time() > deadline:
            print("time box reached at query", j["qid"]); break
        for sysname in ("B", "D", "C"):
            if (j["qid"], sysname) in done:
                continue
            prompt = dict(B=vlm.prompt_direct, C=vlm.prompt_localize)[sysname](j["target"]) if sysname != "D" else vlm.prompt_grounded(j["target"], j["boxes"])
            path = j["path"] if sysname != "D" else grounded_image(j)
            t0 = time.time()
            ans, n_new, size, err, used = "", 0, [0, 0], "oom", None
            for px in [max_pixels] + fallback:
                oom = False
                try:
                    ans, n_new, size = ask(path, prompt, MAX_TOKENS[sysname], px); err, used = None, px
                except torch.cuda.OutOfMemoryError:
                    oom = True
                if not oom:
                    break
                gc.collect(); torch.cuda.empty_cache()     # outside the except block, so the failed step's tensors are really released
            parsed = vlm.parse_points(ans) if sysname == "C" else vlm.parse_int(ans)
            f.write(json.dumps(dict(qid=j["qid"], system=sysname, dataset=j["dataset"], image=j["image"], sequence=j["sequence"], target=j["target"], count=j["count"], parsed=parsed, truncated=n_new >= MAX_TOKENS[sysname],
                                    raw=ans[:6000], seconds=round(time.time() - t0, 2), new_tokens=n_new, input_hw=size, max_pixels_used=used, error=err)) + "\n")
            f.flush()
print("worker done")
"""
Path("dk_vlm_worker.py").write_text(WORKER)

def launch(limit, deadline):
    procs = []
    for i in range(max(N_GPU, 1)):
        gpu = str(i) if CUDA else "cpu"
        log = open(OUT / "logs" / f"vlm_{gpu}.log", "a")
        procs.append((subprocess.Popen([sys.executable, "dk_vlm_worker.py", gpu, str(max(N_GPU, 1)), str(OUT), MODEL_ID, str(MAX_PIXELS), str(deadline), str(limit), ",".join(map(str, FALLBACK_PIXELS))], stdout=log, stderr=subprocess.STDOUT), log))
    for p, log in procs:
        rc = p.wait(); log.close()
        print(Path(log.name).read_text(errors="replace")[-1200:])
        assert rc == 0, f"worker failed: see {log.name}"

def answers():
    rows = [json.loads(l) for p in sorted((OUT / "results").glob("answers_*.jsonl")) for l in p.read_text().splitlines()]
    return pd.DataFrame(rows)

## A few real answers first

In [ ]:
t0 = time.time()
launch(2, time.time() + 1800)
A = answers()
print(A[["system", "target", "count", "parsed", "seconds", "new_tokens", "input_hw", "raw"]].assign(raw=lambda d: d.raw.str[:80]).to_string())
print(f"first answers took {(time.time() - t0) / 60:.1f} min, including loading the model")

## Full run (GPU only)

In [ ]:
t0 = time.time()
if not SMOKE:
    launch(0, t0 + TIME_H * 3600)
A = answers()
A.to_csv(OUT / "results" / "vlm_answers.csv", index=False)
s = A.groupby("system").agg(n=("qid", "size"), parsed=("parsed", lambda v: v.notna().mean()), truncated=("truncated", "mean"), seconds=("seconds", "mean"), total_h=("seconds", lambda v: v.sum() / 3600))
s.to_csv(OUT / "results" / "vlm_run_summary.csv")
print(s.round(3).to_string())
(OUT / "results" / "run_meta.json").write_text(json.dumps(dict(model=MODEL_ID, max_pixels=MAX_PIXELS, quant="4-bit nf4", n_gpu=N_GPU, time_box_h=TIME_H, wall_hours=(time.time() - t0) / 3600, queries_planned=len(Q), queries_done=int(A.qid.nunique()), only_qids=ONLY_QIDS, fallback_pixels=FALLBACK_PIXELS,
    max_tokens=MAX_TOKENS, vd_conf=VD_CONF, uv_conf=UV_CONF, smoke=SMOKE, python=platform.python_version(), torch=torch.__version__), indent=1))
with zipfile.ZipFile(OUT / "send_back.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted((OUT / "results").rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("send_back.zip written;", A.qid.nunique(), "of", len(Q), "queries answered")